<a href="https://colab.research.google.com/github/gcallj/test/blob/main/GA_stock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install deap

In [ ]:
# -*- coding: utf-8 -*-
"""
GA + Walk-Forward ML (OOS) + Intraday (OHLC) Backtest  — v2 (fixed outputs)
============================================================================

Fixes vs previous version:
1) APPLY "best buy" / "best sell" now uses **next-day OHLC** (i+1) as intended.
   - Signal is computed at end of day i (close), and the suggested order is for day i+1.
   - Columns:
        signal_eod          : signal decided at close(i)
        next_day_filled     : whether the limit would be filled on day i+1
        best_buy_value      : filled price (NaN if not filled)
        best_sell_value     : filled price (NaN if not filled)
        entry_ref_price     : best_* if filled else open(i+1) (optional reference)
        stop/take levels are computed from entry_ref_price.

2) "Score & signal in Excel not working" (all holds / all best_buy == close):
   - We compute suggested entry for next day and keep fallback values for rows without next-day data.

3) Too many tickers with 0 trades (TEret=0):
   - GA fitness penalizes strategies with very low trades/exposure (prevents "do nothing" winning).
   - GA search ranges for enter_abs are made more permissive (lower thresholds).

4) Cleaner & richer metrics:
   - WF: AUC mean/std, ACC mean, PR-AUC mean, logloss, brier.
   - Trading: return/mdd/sharpe/trades/exposure/win_rate/avg_trade for GA and TEST.
   - Period print per ticker: train/test date ranges.

Expected columns in HISTORY_CSV:
- Date, ticker, open, high, low, close
- plus numeric feature columns.

Outputs:
- CSV: apply_last_{APPLY_DAYS}d__H{FWD_H}.csv
- XLSX: summary_latest + apply_last_{APPLY_DAYS}d

NOTE
- This is research/backtest code. Not financial advice.

Author: ChatGPT (generated)
"""

from google.colab import drive
drive.mount("/content/drive")

import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from deap import base, creator, tools, algorithms

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss, brier_score_loss
from scipy.stats import spearmanr


# ==============================================================================
# 0) CONFIG
# ==============================================================================
HISTORY_CSV_PATH = "/content/drive/MyDrive/history_consolidated.csv"
OUTPUT_DIR       = "/content/drive/MyDrive/"

DATE_COL   = "Date"
TICKER_COL = "ticker"

OPEN_COL  = "open"
HIGH_COL  = "high"
LOW_COL   = "low"
CLOSE_COL = "close"

ONLY_SA    = True
LONG_ONLY  = False

APPLY_DAYS = 5
FWD_H = 5
TARGET_RET_THRESHOLD = 0.015  # classify only meaningful +1.5% moves
TARGET_ATR_MULT = 0.75        # optional ATR-scaled threshold (effective threshold = max(ret, ATR*mult))

# Regime filter (MA200)
MA_WINDOW = 200
USE_MA_SLOPE_FILTER = True
MA_SLOPE_LOOKBACK = 10  # shorter lookback for more responsive regime detection
MA_SLOPE_EPS = 0.0
REQUIRE_MA_FOR_ENTRY = True
REQUIRE_MA_FOR_SELL_MA = False

# Z-score (normalizes score_full)
EV_CLIP = 5.0
EV_EMA_SPAN = 3

# Probability-to-direction scaling (higher = sharper separation near 0.5)
PROB_DIRECTION_SCALE = 4.0

# ATR
ATR_WINDOW = 14
ATR_MIN_PERIODS = 14
ATR_EPS = 1e-12

# GA ranges
ATR_MULT_RANGE = (1.5, 3.5)
RR_MULT_RANGE  = (1.5, 4.0)  # wider RR to pursue larger trend-following payoffs

# Friction
COST_BPS     = 6.0
SLIPPAGE_BPS = 5.0
COST_PER_TRADE_PCT = 0.0020  # fixed round-trip friction per completed trade

MIN_PRICE     = 0.01
CAP_DAILY_RET = 0.30
CAP_TRADE_RET = 3.00

ONE_YEAR_DAYS = 252
GA_WF_TRAIN_YEARS = 4
GA_WF_TEST_DAYS = 252
GA_WF_STEP_DAYS = 252
LAMBDA_MDD_1Y = 0.70
MAX_EXPOSURE_1Y = 0.70

# GA hyperparams (reduce for speed)
RANDOM_SEED = 42
GA_POP_SIZE = 220
GA_NGEN     = 55
GA_CX_PB    = 0.70
GA_MUT_PB   = 0.40
GA_TOURN    = 3
EARLY_STOP  = 10
GA_WF_SPLITS = 3
GA_MUT_SIGMA_START = 0.14
GA_MUT_SIGMA_END = 0.04
GA_GENE_MUT_PB_START = 0.25
GA_GENE_MUT_PB_END = 0.08
GA_HOF_SIZE = 5
GA_MIN_TRADES_PER_FOLD = 15
GA_OVERTRADING_TRADES_PER_FOLD = 150
GA_FOLD_STABILITY_PENALTY = 1.5
GA_MIN_TRADES_FOR_SIGNIFICANCE = 15

# ML (walk-forward)
WF_SPLITS = 5
ML_RECENCY_HALF_LIFE = 252
ML_RET_CAP = 0.30          # cap fwd return before ATR-normalization
ML_MIN_TRAIN = 260
SCORE_LOOKBACK = 504

# Calibrate score_ev using realized forward-return feedback (per ticker)
USE_RETURN_FEEDBACK_CALIBRATION = True
RETURN_FEEDBACK_BLEND = 0.70
RETURN_FEEDBACK_BLEND_MAX = 0.10
RETURN_FEEDBACK_MIN_ROWS = 300
RETURN_FEEDBACK_BINS = 10
RETURN_FEEDBACK_MIN_IC = 0.04
RETURN_FEEDBACK_TARGET_IC = 0.08
RETURN_FEEDBACK_BIN_SHRINK = 80.0

# Feature selection
MIN_ROWS_TICKER = 350  # enough for MA200 min_periods + some buffer
MIN_FEAT_NONNA_FRAC = 0.60
MIN_FEAT_STD = 1e-12
MIN_VALID_SAMPLES_FOR_CORRELATION = 30
MAX_FEATURES = 30

# Intraday entry (limit) based on signal strength
ENTRY_DISCOUNT_RANGE = (0.0, 0.9)
FAST_PERIOD_RANGE = (3, 10)
SLOW_PERIOD_RANGE = (10, 60)
SCORE_CROSS_MIN_ABS = 0.05
ENTRY_SCORE_TRIGGER_ABS = 0.02
ML_STRONG_SCORE_ABS = 0.08

# Avoid "do nothing" strategies
GA_MIN_TRADES = 5
GA_TARGET_TRADES = 10
GA_MIN_EXPOSURE = 0.05
GA_TRADE_BONUS_PER = 0.015
MAX_TRADES_PER_YEAR = 60
OVERTRADING_PENALTY_PER_TRADE = 0.03
GA_MIN_WF_AUC_TO_RUN = 0.53
GA_MIN_WF_AP_TO_RUN = 0.52
GA_MIN_WF_QUALITY_TO_RUN = 0.30
GA_INTERNAL_EXTRA_TRADE_COST_BPS = 30.0

# Prints
PRINT_EVERY = 1
PRINT_FOLD_DETAILS = False
PRINT_TOP_N = 12
PRINT_TAIL_DETAILS = False
PRINT_ML_METRICS = False
TIME_STOP_BARS = 5
ENTRY_VOL_LOOKBACK = 60
ENTRY_ATR_MAX_MULT = 1.8
TEST_ONLY_PREFIX_C_TICKERS = False
TEST_TICKER_PREFIX = "C"
EVAL_ONLY_TICKER = "CURY3"
GA_VERBOSE_PER_GENERATION = True
GA_VERBOSE_MAX_TRADES_PER_GEN = 80

# ==============================================================================
# 1) DEAP SETUP
# ==============================================================================
def setup_global_deap():
    if not hasattr(creator, "FitnessMax_PT"):
        creator.create("FitnessMax_PT", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual_PT"):
        creator.create("Individual_PT", list, fitness=creator.FitnessMax_PT)

setup_global_deap()

# ==============================================================================
# 2) DATA CLASSES
# ==============================================================================
@dataclass
class Params:
    fast_period: float
    slow_period: float
    atr_mult: float
    rr_mult: float
    entry_discount: float

def sanitize_params(p: Params) -> Params:
    fast_period = int(np.clip(round(float(p.fast_period)), FAST_PERIOD_RANGE[0], FAST_PERIOD_RANGE[1]))
    slow_period = int(np.clip(round(float(p.slow_period)), SLOW_PERIOD_RANGE[0], SLOW_PERIOD_RANGE[1]))
    if slow_period <= fast_period:
        slow_period = min(SLOW_PERIOD_RANGE[1], fast_period + 1)
        if slow_period <= fast_period:
            fast_period = max(FAST_PERIOD_RANGE[0], slow_period - 1)
    atr_mult = float(np.clip(p.atr_mult, ATR_MULT_RANGE[0], ATR_MULT_RANGE[1]))
    rr_mult  = float(np.clip(p.rr_mult,  RR_MULT_RANGE[0],  RR_MULT_RANGE[1]))
    entry_discount = float(np.clip(p.entry_discount, ENTRY_DISCOUNT_RANGE[0], ENTRY_DISCOUNT_RANGE[1]))
    return Params(float(fast_period), float(slow_period), atr_mult, rr_mult, entry_discount)


# ==============================================================================
# 3) HELPERS
# ==============================================================================
def _parse_dates_smart(s: pd.Series) -> pd.Series:
    ss = s.astype(str)
    frac_dash = ss.str.contains("-", regex=False).mean()
    if frac_dash > 0.5:
        return pd.to_datetime(ss, errors="coerce", dayfirst=False)
    return pd.to_datetime(ss, errors="coerce", dayfirst=True)

def _sigmoid(x: float) -> float:
    x = float(np.clip(x, -50, 50))
    return float(1.0 / (1.0 + math.exp(-x)))

def add_sma200(df: pd.DataFrame) -> pd.DataFrame:
    df["sma200"] = df.groupby(TICKER_COL, sort=False)[CLOSE_COL].transform(
        lambda s: s.rolling(MA_WINDOW, min_periods=MA_WINDOW).mean()
    )
    if USE_MA_SLOPE_FILTER:
        df["sma200_slope"] = df.groupby(TICKER_COL, sort=False)["sma200"].transform(
            lambda x: (x - x.shift(MA_SLOPE_LOOKBACK)) / float(MA_SLOPE_LOOKBACK)
        )
    else:
        df["sma200_slope"] = np.nan
    return df

def add_atr_ohlc_fast(df: pd.DataFrame) -> pd.DataFrame:
    """
    ATR fast: compute TR with vector ops + groupby rolling mean.
    TR = max(high-low, abs(high-prev_close), abs(low-prev_close))
    """
    h = df[HIGH_COL].astype(float)
    l = df[LOW_COL].astype(float)
    c = df[CLOSE_COL].astype(float)
    pc = df.groupby(TICKER_COL, sort=False)[CLOSE_COL].shift(1).astype(float)

    tr1 = (h - l).abs()
    tr2 = (h - pc).abs()
    tr3 = (l - pc).abs()

    tr = np.nanmax(np.vstack([tr1.to_numpy(), tr2.to_numpy(), tr3.to_numpy()]), axis=0)
    tr = pd.Series(tr, index=df.index)

    df["atr"] = tr.groupby(df[TICKER_COL], sort=False).transform(
        lambda s: s.rolling(ATR_WINDOW, min_periods=ATR_MIN_PERIODS).mean()
    )
    return df


def add_technical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add robust technical indicators for ML pipeline.
    
    Features (enhanced set):
    - Distance from Moving Average (20-day and 50-day)
    - ROC (Rate of Change, 5-day and 10-day)
    - Volatility (20-day rolling std)
    - Relative Volume (volume vs 20-day average)
    - Stochastic Oscillator (14-day %K and %D)
    - RSI (14-day)
    - MACD histogram
    - Bollinger Band %B
    - CCI (Commodity Channel Index, 20-day)
    - ADX proxy (directional strength)
    - Williams %R
    - Price momentum (rate of change of ROC)
    """
    grouped = df.groupby(TICKER_COL, sort=False)
    
    # Distance from 20-day Moving Average
    df['dist_ma20'] = grouped[CLOSE_COL].transform(
        lambda x: (lambda ma20: (x - ma20) / ma20)(x.rolling(20, min_periods=1).mean())
    )
    
    # Distance from 50-day Moving Average
    df['dist_ma50'] = grouped[CLOSE_COL].transform(
        lambda x: (lambda ma50: (x - ma50) / ma50)(x.rolling(50, min_periods=1).mean())
    )
    
    # ROC (Rate of Change) - 5 day (short-term momentum)
    df['roc_5'] = grouped[CLOSE_COL].transform(
        lambda x: (x - x.shift(5)) / x.shift(5).replace(0, np.nan)
    )
    
    # ROC (Rate of Change) - 10 day
    df['roc_10'] = grouped[CLOSE_COL].transform(
        lambda x: (x - x.shift(10)) / x.shift(10).replace(0, np.nan)
    )
    
    # Volatility - 20-day rolling standard deviation of returns
    df['volatility_20'] = grouped[CLOSE_COL].transform(
        lambda x: x.pct_change().rolling(20, min_periods=1).std()
    )
    
    # Relative Volume - volume relative to 20-day average
    if 'volume' in df.columns:
        df['rel_volume'] = grouped['volume'].transform(
            lambda x: (lambda avg: x / avg.replace(0, np.nan))(x.rolling(20, min_periods=1).mean())
        )
    else:
        df['rel_volume'] = np.nan
    
    # Stochastic Oscillator (14-day %K)
    high_14 = grouped[HIGH_COL].transform(lambda x: x.rolling(14, min_periods=1).max())
    low_14 = grouped[LOW_COL].transform(lambda x: x.rolling(14, min_periods=1).min())
    denom = (high_14 - low_14).replace(0, np.nan)
    df['stochastic_k'] = 100 * (df[CLOSE_COL] - low_14) / denom
    
    # Stochastic %D (3-day SMA of %K)
    df['stochastic_d'] = grouped['stochastic_k'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
    
    # RSI (14-day)
    def _rsi_transform(x):
        delta = x.diff()
        gain = delta.clip(lower=0).rolling(14, min_periods=1).mean()
        loss = (-delta.clip(upper=0)).rolling(14, min_periods=1).mean()
        rs = gain / loss.replace(0, np.nan)
        return 100 - (100 / (1 + rs))
    df['rsi_14'] = grouped[CLOSE_COL].transform(_rsi_transform)
    
    # RSI acceleration (today RSI vs 5 days ago)
    df['rsi_accel_5'] = grouped['rsi_14'].transform(lambda x: x - x.shift(5))
    
    # MACD histogram (12-26-9)
    ema12 = grouped[CLOSE_COL].transform(lambda x: x.ewm(span=12, adjust=False, min_periods=1).mean())
    ema26 = grouped[CLOSE_COL].transform(lambda x: x.ewm(span=26, adjust=False, min_periods=1).mean())
    macd_line = ema12 - ema26
    signal_line = macd_line.groupby(df[TICKER_COL], sort=False).transform(
        lambda x: x.ewm(span=9, adjust=False, min_periods=1).mean()
    )
    df['macd_hist'] = (macd_line - signal_line) / df[CLOSE_COL].replace(0, np.nan)
    
    # Bollinger Band %B (20-day, 2 std)
    bb_ma = grouped[CLOSE_COL].transform(lambda x: x.rolling(20, min_periods=1).mean())
    bb_std = grouped[CLOSE_COL].transform(lambda x: x.rolling(20, min_periods=1).std())
    bb_upper = bb_ma + 2 * bb_std
    bb_lower = bb_ma - 2 * bb_std
    bb_width = (bb_upper - bb_lower).replace(0, np.nan)
    df['bb_pctb'] = (df[CLOSE_COL] - bb_lower) / bb_width
    
    # CCI (Commodity Channel Index, 20-day)
    tp = (df[HIGH_COL] + df[LOW_COL] + df[CLOSE_COL]) / 3.0
    tp_ma = tp.groupby(df[TICKER_COL], sort=False).transform(lambda x: x.rolling(20, min_periods=1).mean())
    tp_md = tp.groupby(df[TICKER_COL], sort=False).transform(lambda x: x.rolling(20, min_periods=1).apply(lambda w: np.mean(np.abs(w - w.mean())), raw=True))
    df['cci_20'] = (tp - tp_ma) / (0.015 * tp_md.replace(0, np.nan))
    
    # ADX proxy: absolute directional movement normalized by ATR
    plus_dm = (df[HIGH_COL] - df[HIGH_COL].groupby(df[TICKER_COL], sort=False).shift(1)).clip(lower=0)
    minus_dm = (df[LOW_COL].groupby(df[TICKER_COL], sort=False).shift(1) - df[LOW_COL]).clip(lower=0)
    plus_di = plus_dm.groupby(df[TICKER_COL], sort=False).transform(lambda x: x.rolling(14, min_periods=1).mean())
    minus_di = minus_dm.groupby(df[TICKER_COL], sort=False).transform(lambda x: x.rolling(14, min_periods=1).mean())
    di_sum = (plus_di + minus_di).replace(0, np.nan)
    df['adx_proxy'] = (plus_di - minus_di).abs() / di_sum
    
    # Williams %R (14-day)
    df['williams_r'] = -100 * (high_14 - df[CLOSE_COL]) / denom
    
    # Price momentum (ROC of ROC: acceleration)
    df['momentum_accel'] = grouped['roc_10'].transform(
        lambda x: x - x.shift(5)
    )

    # Volatilidade relativa (ATR / Close)
    df['vol_rel_atr'] = df['atr'] / df[CLOSE_COL].replace(0, np.nan)

    # Distância da média longa (Close vs SMA200)
    if 'sma200' in df.columns:
        df['dist_sma200'] = (df[CLOSE_COL] - df['sma200']) / df['sma200'].replace(0, np.nan)
    else:
        df['dist_sma200'] = np.nan

    # Padrão de volume (volume / média de 20 dias)
    df['volume_pattern_20'] = df['rel_volume']

    # Regime de mercado proxy (inclinação da SMA200)
    if 'sma200_slope' in df.columns:
        df['regime_sma200_slope'] = df['sma200_slope']
        df['regime_bull'] = (df['sma200_slope'] > 0).astype(float)
    else:
        df['regime_sma200_slope'] = np.nan
        df['regime_bull'] = np.nan
    
    return df

def regime_ok(sig: int, price: float, sma200: float, sma_slope: float, ml_score: float = np.nan) -> bool:
    strong_buy = np.isfinite(ml_score) and (float(ml_score) >= ML_STRONG_SCORE_ABS)
    decent_signal = np.isfinite(ml_score) and (abs(float(ml_score)) >= ENTRY_SCORE_TRIGGER_ABS)

    # If SMA200 is not available, allow entry for decent ML signals
    if sig > 0:
        if (sma200 is None) or (not np.isfinite(sma200)):
            return decent_signal
        if (price < sma200) and (not strong_buy) and (not decent_signal):
            return False

    # MA rule: relaxed — decent signals can override MA filter
    if sig > 0:
        if REQUIRE_MA_FOR_ENTRY:
            if (sma200 is None) or (not np.isfinite(sma200)):
                ok_ma = decent_signal
            else:
                ok_ma = (price >= sma200) or strong_buy or decent_signal
        else:
            ok_ma = True
    else:
        if REQUIRE_MA_FOR_SELL_MA and REQUIRE_MA_FOR_ENTRY:
            if (sma200 is None) or (not np.isfinite(sma200)):
                ok_ma = decent_signal
            else:
                ok_ma = (price < sma200) or decent_signal
        else:
            ok_ma = True

    # slope rule: relaxed — decent signals bypass slope filter
    if USE_MA_SLOPE_FILTER:
        if (sma_slope is None) or (not np.isfinite(sma_slope)):
            ok_sl = decent_signal
        elif sig > 0 and (strong_buy or decent_signal):
            ok_sl = True
        else:
            ok_sl = (sma_slope > MA_SLOPE_EPS) if sig > 0 else (sma_slope < -MA_SLOPE_EPS)
    else:
        ok_sl = True

    return bool(ok_ma and ok_sl)

def buyhold_capped(close: np.ndarray) -> float:
    n = len(close)
    if n < 2:
        return 0.0
    log_eq = 0.0
    for i in range(1, n):
        pr0, pr1 = float(close[i-1]), float(close[i])
        if pr0 <= MIN_PRICE or pr1 <= MIN_PRICE:
            continue
        daily = (pr1/pr0) - 1.0
        daily = float(np.clip(daily, -CAP_DAILY_RET, CAP_DAILY_RET))
        log_eq += math.log1p(daily)
    return float(math.exp(log_eq) - 1.0)

def fitness_return_1y(stats_1y: Dict[str, float]) -> float:
    ret = float(stats_1y["total_return"])
    mdd_abs = abs(float(stats_1y["mdd"]))
    expo = float(stats_1y["exposure"])
    n_tr = float(stats_1y["n_trades"])
    sh = float(stats_1y["sharpe"])
    wr = float(stats_1y.get("win_rate", 0.0))
    n_days = float(stats_1y.get("n_days", ONE_YEAR_DAYS))

    f = 1.00 * ret - 0.65 * mdd_abs + 0.08 * sh

    if n_days >= (8.0 * ONE_YEAR_DAYS):
        min_required = 1.0 * (n_days / ONE_YEAR_DAYS)
        if n_tr < min_required:
            return -100.0

    if expo > MAX_EXPOSURE_1Y:
        f -= (expo - MAX_EXPOSURE_1Y) * 1.5
    if expo < GA_MIN_EXPOSURE:
        f -= (GA_MIN_EXPOSURE - expo) * 2.0

    avg_tr = float(stats_1y.get("avg_trade", 0.0))
    years = max(1e-9, n_days / ONE_YEAR_DAYS)
    trades_per_year = n_tr / years
    if trades_per_year > MAX_TRADES_PER_YEAR:
        f -= OVERTRADING_PENALTY_PER_TRADE * (trades_per_year - MAX_TRADES_PER_YEAR)

    if n_tr < GA_MIN_TRADES:
        deficit = GA_MIN_TRADES - n_tr
        f -= 0.08 * deficit + 0.04 * (deficit ** 2)
    elif n_tr < GA_TARGET_TRADES:
        # soft penalty band until statistically safer trade count
        gap = GA_TARGET_TRADES - n_tr
        f -= 0.01 * gap

    f += min(n_tr, 30.0) * GA_TRADE_BONUS_PER
    if n_tr >= GA_MIN_TRADES:
        f += max(0.0, wr - 0.5) * 0.12
    if (n_tr >= 20.0) and (wr < 0.45):
        f -= (0.45 - wr) * 0.40

    avg_tr_clip = float(np.clip(avg_tr, -0.03, 0.03))
    f += avg_tr_clip * 0.80
    if avg_tr_clip < 0.0:
        f -= min(0.20, abs(avg_tr_clip) * 4.0)
    else:
        f += min(0.12, avg_tr_clip * 3.0)

    return float(f)

def ema_np(x: np.ndarray, span: int) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64)
    out = np.full(len(arr), np.nan, dtype=np.float64)
    if len(arr) == 0:
        return out
    alpha = 2.0 / (float(max(1, span)) + 1.0)
    prev = np.nan
    for i, v in enumerate(arr):
        if not np.isfinite(v):
            out[i] = prev if np.isfinite(prev) else np.nan
            continue
        if not np.isfinite(prev):
            prev = float(v)
        else:
            prev = alpha * float(v) + (1.0 - alpha) * prev
        out[i] = prev
    return out

def make_signal_eod(score_ev_series: np.ndarray, i: int, p: Params, close_eod: float, sma200_eod: float, slope_eod: float) -> str:
    """
    Signal decided at end of day i (to be acted on day i+1) using EMA crossover.
    """
    if (i <= 0) or (i >= len(score_ev_series)):
        return "hold"
    ev_i = float(score_ev_series[i]) if np.isfinite(score_ev_series[i]) else np.nan
    if (not np.isfinite(ev_i)) or (not np.isfinite(close_eod)):
        return "hold"

    fast = ema_np(score_ev_series[:i+1], int(p.fast_period))
    if (not np.isfinite(fast[i])) or (not np.isfinite(fast[i-1])):
        return "hold"
    entry_up = (fast[i-1] <= ENTRY_SCORE_TRIGGER_ABS) and (fast[i] > ENTRY_SCORE_TRIGGER_ABS)
    entry_dn = (fast[i-1] >= -ENTRY_SCORE_TRIGGER_ABS) and (fast[i] < -ENTRY_SCORE_TRIGGER_ABS)

    if entry_up and (ev_i > ENTRY_SCORE_TRIGGER_ABS) and regime_ok(+1, close_eod, sma200_eod, slope_eod, ev_i):
        return "buy"
    if (not LONG_ONLY) and entry_dn and (ev_i < -ENTRY_SCORE_TRIGGER_ABS) and regime_ok(-1, close_eod, sma200_eod, slope_eod, ev_i):
        return "sell"
    return "hold"

def score_0_100_from_ev(
    score_ev: float,
    recent_scores_ev: np.ndarray,
    quality: float,
) -> float:
    if (not np.isfinite(score_ev)):
        return 50.0
    recent = recent_scores_ev[np.isfinite(recent_scores_ev)] if recent_scores_ev is not None else np.array([], dtype=np.float64)
    if len(recent) == 0:
        pct_rank = 0.5
    else:
        pct_rank = float(np.mean(recent <= score_ev))
    base_score = 1.0 + 98.0 * pct_rank
    tilt_factor = 0.35 + 0.65 * float(np.clip(quality, 0.0, 1.0))
    score = 50.0 + tilt_factor * (base_score - 50.0)
    return float(np.clip(score, 0.0, 100.0))

def compute_quality_factor(test_sharpe: float, test_return: float, trades_1y: float) -> float:
    q_sh = _sigmoid((float(test_sharpe) - 0.10) / 0.30) if np.isfinite(test_sharpe) else 0.5
    q_ret = _sigmoid(float(test_return) / 0.15) if np.isfinite(test_return) else 0.5
    q_tr = _sigmoid((float(trades_1y) - 8.0) / 4.0) if np.isfinite(trades_1y) else 0.3
    return float(np.clip(0.45 * q_sh + 0.35 * q_ret + 0.20 * q_tr, 0.0, 1.0))

def compute_wf_quality(wf_auc_mean: float, wf_ap_mean: float, wf_auc_std: float) -> float:
    q_auc = np.clip((float(wf_auc_mean) - 0.50) / 0.18, 0.0, 1.0) if np.isfinite(wf_auc_mean) else 0.0
    q_ap = np.clip((float(wf_ap_mean) - 0.50) / 0.20, 0.0, 1.0) if np.isfinite(wf_ap_mean) else 0.0
    q_stab = 1.0 - np.clip(float(wf_auc_std) / 0.12, 0.0, 1.0) if np.isfinite(wf_auc_std) else 0.0
    return float(np.clip(0.45 * q_auc + 0.35 * q_ap + 0.20 * q_stab, 0.0, 1.0))

def adjust_params_by_wf_quality(p: Params, wf_quality: float) -> Params:
    p = sanitize_params(p)
    q = float(np.clip(wf_quality, 0.0, 1.0))
    # weaker models ask better prices and slightly smoother crossover windows
    entry_discount = p.entry_discount * (1.0 + (1.0 - q) * 0.60)
    fast_period = int(round(p.fast_period + (1.0 - q) * 1.5))
    slow_period = int(round(p.slow_period + (1.0 - q) * 4.0))
    return sanitize_params(Params(float(fast_period), float(slow_period), p.atr_mult, p.rr_mult, entry_discount))

def compute_model_entry_price(o1: float, atr1: float, score_ev_eod: float, enter_abs: float, side: int, entry_discount: float) -> float:
    if (not np.isfinite(o1)) or (not np.isfinite(atr1)) or atr1 <= ATR_EPS:
        return float("nan")
    if (not np.isfinite(score_ev_eod)) or (not np.isfinite(enter_abs)) or enter_abs <= 0:
        return float("nan")

    strength_ratio = abs(float(score_ev_eod)) / float(enter_abs)
    strength_ratio = max(strength_ratio, 0.5)
    inv_strength = float(np.clip(1.0 / strength_ratio, 0.3, 2.0))
    discount_atr = float(entry_discount * inv_strength * atr1)
    return float(o1 - discount_atr) if side > 0 else float(o1 + discount_atr)

def compute_levels_from_atr(entry_price: float, atr_val: float, p: Params):
    """
    Returns: stop_abs, take_abs, stop_pct, take_pct, buy_entry, buy_stop, buy_take, sell_entry, sell_stop, sell_take
    """
    if (not np.isfinite(entry_price)) or (entry_price <= 0) or (not np.isfinite(atr_val)) or (atr_val <= ATR_EPS):
        return (np.nan, np.nan, np.nan, np.nan,
                np.nan, np.nan, np.nan,
                np.nan, np.nan, np.nan)

    p = sanitize_params(p)
    stop_abs = float(p.atr_mult * atr_val)
    take_abs = float(p.rr_mult  * stop_abs)

    stop_pct = float(stop_abs / max(entry_price, 1e-12))
    take_pct = float(take_abs / max(entry_price, 1e-12))

    buy_entry = float(entry_price)
    buy_stop  = float(entry_price - stop_abs)
    buy_take  = float(entry_price + take_abs)

    sell_entry = float(entry_price)
    sell_stop  = float(entry_price + stop_abs)
    sell_take  = float(entry_price - take_abs)

    return (stop_abs, take_abs, stop_pct, take_pct,
            buy_entry, buy_stop, buy_take,
            sell_entry, sell_stop, sell_take)

def nextday_limit_fill(o1: float, h1: float, l1: float, atr1: float, score_ev_eod: float, enter_abs: float, side: int, entry_discount: float) -> Tuple[bool, float, float]:
    if (not np.isfinite(o1)) or (not np.isfinite(h1)) or (not np.isfinite(l1)) or (not np.isfinite(atr1)) or atr1 <= ATR_EPS:
        return (False, np.nan, np.nan)

    limit = compute_model_entry_price(o1, atr1, score_ev_eod, enter_abs, side, entry_discount)
    if not np.isfinite(limit):
        return (False, np.nan, np.nan)

    if side > 0:
        if float(o1) <= limit:
            return (True, float(o1), float(limit))
        if float(l1) <= limit <= float(h1):
            return (True, float(limit), float(limit))
        return (False, np.nan, float(limit))
    else:
        if float(o1) >= limit:
            return (True, float(o1), float(limit))
        if float(l1) <= limit <= float(h1):
            return (True, float(limit), float(limit))
        return (False, np.nan, float(limit))

def make_recency_weights(n: int, half_life: int) -> np.ndarray:
    if n <= 1:
        return np.ones(n, dtype=np.float64)
    lam = math.log(2.0) / max(1.0, float(half_life))
    idx = np.arange(n, dtype=np.float64)
    w = np.exp(-lam * ((n - 1) - idx))
    w = w / max(1e-12, float(np.mean(w)))
    return w.astype(np.float64)

def probability_to_direction_sign(p: np.ndarray) -> np.ndarray:
    # compress low-confidence probs around 0.5 and keep high-confidence tails
    x = np.asarray(p, dtype=np.float64) - 0.5
    sign = np.tanh(PROB_DIRECTION_SCALE * x)
    return np.clip(sign, -1.0, 1.0)

def _safe_pr_auc(y_true_bin: np.ndarray, y_prob: np.ndarray) -> float:
    try:
        from sklearn.metrics import average_precision_score
        return float(average_precision_score(y_true_bin, y_prob))
    except Exception:
        return float("nan")


# ==============================================================================
# 4) ML (Walk-Forward OOS probabilities)
# ==============================================================================
def get_clean_walk_forward_predictions(
    X: np.ndarray,
    y_bin: np.ndarray,
    y_mag: np.ndarray,
    w: np.ndarray,
    dates: np.ndarray,
    ticker: str,
    n_splits: int = WF_SPLITS
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, float]]:
    tscv = TimeSeriesSplit(n_splits=n_splits)

    oos_prob = np.full(len(y_bin), np.nan, dtype=np.float64)
    oos_mag = np.full(len(y_bin), np.nan, dtype=np.float64)

    aucs, accs, briers, loglosses, ap_scores = [], [], [], [], []
    fold_ranges = []

    for fold_i, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
        X_tr, y_tr = X[train_idx], y_bin[train_idx]
        X_te, y_te = X[test_idx], y_bin[test_idx]
        mag_tr = y_mag[train_idx]
        w_tr = w[train_idx] if w is not None else None

        clf = HistGradientBoostingClassifier(
            learning_rate=0.03,
            max_iter=400,
            max_depth=4,
            min_samples_leaf=25,
            max_leaf_nodes=31,
            l2_regularization=2.0,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=15,
            random_state=RANDOM_SEED,
        )
        reg = HistGradientBoostingRegressor(
            learning_rate=0.03,
            max_iter=400,
            max_depth=4,
            min_samples_leaf=25,
            max_leaf_nodes=31,
            l2_regularization=2.0,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=15,
            random_state=RANDOM_SEED,
        )
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        reg.fit(X_tr, mag_tr, sample_weight=w_tr)

        p_te = clf.predict_proba(X_te)[:, 1].astype(np.float64)
        m_te = np.clip(reg.predict(X_te).astype(np.float64), 0.0, None)
        oos_prob[test_idx] = p_te
        oos_mag[test_idx] = m_te

        train_start = pd.to_datetime(dates[train_idx[0]]).date()
        train_end = pd.to_datetime(dates[train_idx[-1]]).date()
        test_start = pd.to_datetime(dates[test_idx[0]]).date()
        test_end = pd.to_datetime(dates[test_idx[-1]]).date()
        fold_msg = f"[{ticker}] Fold {fold_i}/{n_splits}: TRAIN {train_start} -> {train_end} ({len(train_idx)} rows) | TEST {test_start} -> {test_end} ({len(test_idx)} rows)"
        if PRINT_FOLD_DETAILS:
            print("  " + fold_msg)
        fold_ranges.append(f"{train_start}|{train_end}|{test_start}|{test_end}")

        try:
            auc = roc_auc_score(y_te, p_te)
        except Exception:
            auc = 0.5
        yhat = (p_te >= 0.5).astype(int)
        acc = accuracy_score(y_te, yhat)
        try:
            ll = log_loss(y_te, np.clip(p_te, 1e-6, 1-1e-6))
        except Exception:
            ll = float("nan")
        try:
            br = brier_score_loss(y_te, p_te)
        except Exception:
            br = float("nan")
        ap = _safe_pr_auc(y_te, p_te)

        aucs.append(float(auc)); accs.append(float(acc)); loglosses.append(float(ll)); briers.append(float(br)); ap_scores.append(float(ap) if np.isfinite(ap) else float("nan"))

    direction_sign = probability_to_direction_sign(oos_prob)
    oos_ev = direction_sign * oos_mag

    metrics = {
        "wf_auc_mean": float(np.nanmean(aucs)) if len(aucs) else float("nan"),
        "wf_auc_std":  float(np.nanstd(aucs)) if len(aucs) else float("nan"),
        "wf_acc_mean": float(np.nanmean(accs)) if len(accs) else float("nan"),
        "wf_logloss":  float(np.nanmean(loglosses)) if len(loglosses) else float("nan"),
        "wf_brier":    float(np.nanmean(briers)) if len(briers) else float("nan"),
        "wf_ap_mean":  float(np.nanmean(ap_scores)) if len(ap_scores) else float("nan"),
        "wf_n_folds":  int(len(aucs)),
        "wf_fold_ranges": ';'.join(fold_ranges),
    }
    return oos_ev, oos_prob, oos_mag, metrics


def predict_tail_rows(
    X_train: np.ndarray,
    y_bin_train: np.ndarray,
    y_mag_train: np.ndarray,
    w_train: np.ndarray,
    X_tail: np.ndarray,
    ticker: str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_tail = len(X_tail)
    if n_tail == 0 or len(X_train) < ML_MIN_TRAIN:
        return (np.full(n_tail, np.nan), np.full(n_tail, np.nan), np.full(n_tail, np.nan))

    if PRINT_TAIL_DETAILS:
        print(f"  [{ticker}] Final model: training on {len(X_train)} rows, predicting {n_tail} tail rows")

    clf = HistGradientBoostingClassifier(
        learning_rate=0.03, max_iter=400, max_depth=4,
        min_samples_leaf=25, max_leaf_nodes=31,
        l2_regularization=2.0, early_stopping=True,
        validation_fraction=0.15, n_iter_no_change=15,
        random_state=RANDOM_SEED,
    )
    clf.fit(X_train, y_bin_train, sample_weight=w_train)
    p_tail = clf.predict_proba(X_tail)[:, 1].astype(np.float64)

    reg = HistGradientBoostingRegressor(
        learning_rate=0.03, max_iter=400, max_depth=4,
        min_samples_leaf=25, max_leaf_nodes=31,
        l2_regularization=2.0, early_stopping=True,
        validation_fraction=0.15, n_iter_no_change=15,
        random_state=RANDOM_SEED,
    )
    reg.fit(X_train, y_mag_train, sample_weight=w_train)
    mag_tail = np.clip(reg.predict(X_tail), 0.0, None)

    direction_sign = probability_to_direction_sign(p_tail)
    ev_tail = direction_sign * mag_tail
    return ev_tail, p_tail, mag_tail


# ==============================================================================
# 5) BACKTEST (Intraday OHLC-aware)
# ==============================================================================


def backtest_stats_only_intraday(
    o, h, l, c, score_ev, sma200, sma_slope, atr, p: Params,
    return_trades: bool = False,
    dates: np.ndarray = None,
) -> Dict[str, float]:
    """
    Trade enters on day i using EMA-crossover signal computed from score_ev up to i-1.
    If return_trades=True, includes detailed trade logs in key "trades".
    """
    p = sanitize_params(p)
    n = len(c)
    if n < 3:
        base = {"total_return":0.0,"mdd":0.0,"sharpe":0.0,"n_trades":0.0,"win_rate":0.0,"avg_trade":0.0,"exposure":0.0}
        if return_trades:
            base["trades"] = []
        return base

    score_fast = ema_np(score_ev, int(p.fast_period))
    score_slow = ema_np(score_ev, int(p.slow_period))

    cost_leg = (1.0 - (COST_BPS + SLIPPAGE_BPS)/10000.0)
    log_cost = math.log(max(cost_leg, 1e-12))

    log_eq   = np.zeros(n, dtype=np.float64)
    log_rets = np.zeros(n, dtype=np.float64)

    pos = 0
    entry_price = 0.0
    stop_abs = 0.0
    take_abs = 0.0

    n_trades = 0
    n_wins = 0
    trade_sum = 0.0
    trade_rets = []
    trade_max_favs = []
    pos_days = 0
    bars_in_pos = 0
    cur_max_fav = 0.0
    regime_checks_n = 0
    regime_filtered_n = 0

    trade_logs: List[Dict[str, float]] = []
    cur_trade = None

    def _date_at(idx: int) -> str:
        if dates is None:
            return str(idx)
        try:
            return str(pd.to_datetime(dates[idx]).date())
        except Exception:
            return str(idx)

    for i in range(1, n):
        o1 = float(o[i]); h1 = float(h[i]); l1 = float(l[i]); c1 = float(c[i])
        c0 = float(c[i-1])

        if c0 <= MIN_PRICE or c1 <= MIN_PRICE:
            log_eq[i] = log_eq[i-1]
            continue

        daily = (c1/c0) - 1.0
        daily = float(np.clip(daily, -CAP_DAILY_RET, CAP_DAILY_RET))
        step_ret = (pos * daily) if pos != 0 else 0.0
        log_eq[i] = log_eq[i-1] + math.log1p(step_ret)
        log_rets[i] = log_eq[i] - log_eq[i-1]
        if pos != 0:
            pos_days += 1
            bars_in_pos += 1

        f0 = float(score_fast[i-1]) if np.isfinite(score_fast[i-1]) else np.nan
        s0 = float(score_slow[i-1]) if np.isfinite(score_slow[i-1]) else np.nan
        f1 = float(score_fast[i]) if np.isfinite(score_fast[i]) else np.nan
        s1 = float(score_slow[i]) if np.isfinite(score_slow[i]) else np.nan
        cross_dn_now = np.isfinite(f0) and np.isfinite(s0) and np.isfinite(f1) and np.isfinite(s1) and (f0 >= s0) and (f1 < s1)
        cross_up_now = np.isfinite(f0) and np.isfinite(s0) and np.isfinite(f1) and np.isfinite(s1) and (f0 <= s0) and (f1 > s1)

        if pos > 0:
            cur_max_fav = max(cur_max_fav, float((h1 / max(entry_price, 1e-12)) - 1.0), float((o1 / max(entry_price, 1e-12)) - 1.0))
        elif pos < 0:
            cur_max_fav = max(cur_max_fav, float((entry_price / max(l1, 1e-12)) - 1.0), float((entry_price / max(o1, 1e-12)) - 1.0))

        if pos != 0:
            exited = False
            exit_px = np.nan
            exit_reason = ""

            if pos > 0:
                stop_px = entry_price - stop_abs
                take_px = entry_price + take_abs

                if cross_dn_now:
                    exited, exit_px, exit_reason = True, c1, "cross_dn"
                elif (bars_in_pos >= TIME_STOP_BARS) and (c1 <= entry_price):
                    atr_i = float(atr[i]) if np.isfinite(atr[i]) else np.nan
                    fs, fill_s, _ = nextday_limit_fill(o1, h1, l1, atr_i, -SCORE_CROSS_MIN_ABS, SCORE_CROSS_MIN_ABS, -1, p.entry_discount)
                    exit_candidate = float(fill_s) if fs else c1
                    exited, exit_px, exit_reason = True, exit_candidate, "time_stop"
                elif o1 <= stop_px:
                    exited, exit_px, exit_reason = True, o1, "gap_stop"
                elif o1 >= take_px:
                    exited, exit_px, exit_reason = True, o1, "gap_take"
                else:
                    hit_stop = (l1 <= stop_px)
                    hit_take = (h1 >= take_px)
                    if hit_stop and hit_take:
                        exited, exit_px, exit_reason = True, stop_px, "stop_and_take_same_bar"
                    elif hit_stop:
                        exited, exit_px, exit_reason = True, stop_px, "stop_hit"
                    elif hit_take:
                        exited, exit_px, exit_reason = True, take_px, "take_hit"

                if not exited and REQUIRE_MA_FOR_ENTRY:
                    ma_ = float(sma200[i])
                    if np.isfinite(ma_) and (c1 < ma_):
                        exited, exit_px, exit_reason = True, c1, "ma_filter_exit"

                if exited:
                    trade_ret = (exit_px / max(entry_price, 1e-12)) - 1.0
                    trade_ret = float(np.clip(trade_ret, -CAP_TRADE_RET, CAP_TRADE_RET))
                    log_eq[i] += log_cost
                    log_rets[i] = log_eq[i] - log_eq[i-1]
                    net_trade = (1.0 + trade_ret) * (cost_leg**2) - 1.0
                    net_trade -= COST_PER_TRADE_PCT
                    n_trades += 1
                    if net_trade > 0:
                        n_wins += 1
                    trade_sum += net_trade
                    trade_rets.append(float(net_trade))
                    trade_max_favs.append(float(max(0.0, cur_max_fav)))
                    if return_trades and cur_trade is not None:
                        cur_trade.update({
                            "exit_idx": int(i),
                            "exit_date": _date_at(i),
                            "exit_price": float(exit_px),
                            "exit_reason": exit_reason,
                            "gross_ret": float(trade_ret),
                            "net_ret": float(net_trade),
                            "bars_held": int(bars_in_pos),
                            "max_fav_pct": float(max(0.0, cur_max_fav)),
                        })
                        trade_logs.append(cur_trade)
                        cur_trade = None
                    pos = 0
                    entry_price = 0.0
                    stop_abs = 0.0
                    take_abs = 0.0
                    bars_in_pos = 0
                    cur_max_fav = 0.0
                    continue
            else:
                stop_px = entry_price + stop_abs
                take_px = entry_price - take_abs

                if cross_up_now:
                    exited, exit_px, exit_reason = True, c1, "cross_up"
                elif (bars_in_pos >= TIME_STOP_BARS) and (c1 >= entry_price):
                    atr_i = float(atr[i]) if np.isfinite(atr[i]) else np.nan
                    fb, fill_b, _ = nextday_limit_fill(o1, h1, l1, atr_i, SCORE_CROSS_MIN_ABS, SCORE_CROSS_MIN_ABS, +1, p.entry_discount)
                    exit_candidate = float(fill_b) if fb else c1
                    exited, exit_px, exit_reason = True, exit_candidate, "time_stop"
                elif o1 >= stop_px:
                    exited, exit_px, exit_reason = True, o1, "gap_stop"
                elif o1 <= take_px:
                    exited, exit_px, exit_reason = True, o1, "gap_take"
                else:
                    hit_stop = (h1 >= stop_px)
                    hit_take = (l1 <= take_px)
                    if hit_stop and hit_take:
                        exited, exit_px, exit_reason = True, stop_px, "stop_and_take_same_bar"
                    elif hit_stop:
                        exited, exit_px, exit_reason = True, stop_px, "stop_hit"
                    elif hit_take:
                        exited, exit_px, exit_reason = True, take_px, "take_hit"

                if not exited and REQUIRE_MA_FOR_ENTRY:
                    ma_ = float(sma200[i])
                    if np.isfinite(ma_) and (c1 > ma_):
                        exited, exit_px, exit_reason = True, c1, "ma_filter_exit"

                if exited:
                    trade_ret = (entry_price / max(exit_px, 1e-12)) - 1.0
                    trade_ret = float(np.clip(trade_ret, -CAP_TRADE_RET, CAP_TRADE_RET))
                    log_eq[i] += log_cost
                    log_rets[i] = log_eq[i] - log_eq[i-1]
                    net_trade = (1.0 + trade_ret) * (cost_leg**2) - 1.0
                    net_trade -= COST_PER_TRADE_PCT
                    n_trades += 1
                    if net_trade > 0:
                        n_wins += 1
                    trade_sum += net_trade
                    trade_rets.append(float(net_trade))
                    trade_max_favs.append(float(max(0.0, cur_max_fav)))
                    if return_trades and cur_trade is not None:
                        cur_trade.update({
                            "exit_idx": int(i),
                            "exit_date": _date_at(i),
                            "exit_price": float(exit_px),
                            "exit_reason": exit_reason,
                            "gross_ret": float(trade_ret),
                            "net_ret": float(net_trade),
                            "bars_held": int(bars_in_pos),
                            "max_fav_pct": float(max(0.0, cur_max_fav)),
                        })
                        trade_logs.append(cur_trade)
                        cur_trade = None
                    pos = 0
                    entry_price = 0.0
                    stop_abs = 0.0
                    take_abs = 0.0
                    bars_in_pos = 0
                    cur_max_fav = 0.0
                    continue

        if pos == 0 and i >= 2:
            ev_prev = float(score_ev[i-1]) if np.isfinite(score_ev[i-1]) else np.nan
            f_prev2 = float(score_fast[i-2]) if np.isfinite(score_fast[i-2]) else np.nan
            f_prev = float(score_fast[i-1]) if np.isfinite(score_fast[i-1]) else np.nan

            sig = 0
            entry_trigger = ""
            if np.isfinite(ev_prev) and np.isfinite(f_prev2) and np.isfinite(f_prev):
                entry_up_prev = (f_prev2 <= ENTRY_SCORE_TRIGGER_ABS) and (f_prev > ENTRY_SCORE_TRIGGER_ABS)
                entry_dn_prev = (f_prev2 >= -ENTRY_SCORE_TRIGGER_ABS) and (f_prev < -ENTRY_SCORE_TRIGGER_ABS)
                if entry_up_prev and (ev_prev > ENTRY_SCORE_TRIGGER_ABS):
                    sig = 1
                    entry_trigger = "entry_cross_up"
                elif (not LONG_ONLY) and entry_dn_prev and (ev_prev < -ENTRY_SCORE_TRIGGER_ABS):
                    sig = -1
                    entry_trigger = "entry_cross_down"

            if sig != 0:
                c_prev = float(c[i-1])
                ma_prev = float(sma200[i-1]) if np.isfinite(sma200[i-1]) else np.nan
                sl_prev = float(sma_slope[i-1]) if np.isfinite(sma_slope[i-1]) else np.nan
                regime_checks_n += 1
                if not regime_ok(sig, c_prev, ma_prev, sl_prev, ev_prev):
                    regime_filtered_n += 1
                    continue

                atr_i = float(atr[i]) if np.isfinite(atr[i]) else np.nan
                if (not np.isfinite(atr_i)) or atr_i <= ATR_EPS:
                    continue

                filled, fill_px, _ = nextday_limit_fill(o1, h1, l1, atr_i, ev_prev, ENTRY_SCORE_TRIGGER_ABS, sig, p.entry_discount)
                if not filled:
                    continue

                pos = sig
                entry_price = float(fill_px)
                stop_abs = float(p.atr_mult * atr_i)
                take_abs = float(p.rr_mult * stop_abs)
                bars_in_pos = 0
                cur_max_fav = 0.0
                if return_trades:
                    cur_trade = {
                        "side": "LONG" if sig > 0 else "SHORT",
                        "entry_idx": int(i),
                        "entry_date": _date_at(i),
                        "entry_price": float(entry_price),
                        "entry_trigger": entry_trigger,
                        "entry_ev_prev": float(ev_prev),
                        "entry_score_fast_prev2": float(f_prev2),
                        "entry_score_fast_prev": float(f_prev),
                        "atr": float(atr_i),
                        "stop_abs": float(stop_abs),
                        "take_abs": float(take_abs),
                        "entry_discount": float(p.entry_discount),
                    }
                log_eq[i] += log_cost
                log_rets[i] = log_eq[i] - log_eq[i-1]

    exposure = float(pos_days / max(n-1, 1))
    total_return = float(math.exp(log_eq[-1] - log_eq[0]) - 1.0)

    peak = np.maximum.accumulate(log_eq)
    dd = np.exp(log_eq - peak) - 1.0
    mdd = float(np.min(dd))

    mu = float(np.nanmean(log_rets))
    sd = float(np.nanstd(log_rets, ddof=1))
    sharpe = (mu / sd) * math.sqrt(252.0) if (sd > 1e-9) else 0.0
    downside = log_rets[log_rets < 0.0]
    downside_sd = float(np.nanstd(downside, ddof=1)) if len(downside) > 1 else 0.01
    sortino = (mu / max(downside_sd, 1e-9)) * math.sqrt(252.0)

    win_rate = (n_wins / n_trades) if n_trades > 0 else 0.0
    avg_trade = (trade_sum / n_trades) if n_trades > 0 else 0.0
    trade_std = float(np.nanstd(np.asarray(trade_rets, dtype=np.float64), ddof=1)) if len(trade_rets) > 1 else float("nan")
    max_fav_pct = float(np.mean(trade_max_favs)) if len(trade_max_favs) > 0 else 0.0

    peak_log = np.maximum.accumulate(log_eq)
    underwater = (log_eq < (peak_log - 1e-12)).astype(np.int32)
    cur_dd_dur = 0
    max_dd_dur = 0
    for flag in underwater:
        if flag:
            cur_dd_dur += 1
            if cur_dd_dur > max_dd_dur:
                max_dd_dur = cur_dd_dur
        else:
            cur_dd_dur = 0

    regime_filtered_pct = (float(regime_filtered_n) / float(regime_checks_n)) if regime_checks_n > 0 else 0.0

    n_longs = float(np.sum([1 for t in trade_logs if t.get("side") == "LONG"])) if return_trades else float("nan")
    n_shorts = float(np.sum([1 for t in trade_logs if t.get("side") == "SHORT"])) if return_trades else float("nan")

    out = {
        "total_return": total_return,
        "mdd": mdd,
        "sharpe": sharpe,
        "sortino": float(sortino),
        "n_trades": float(n_trades),
        "n_longs": n_longs,
        "n_shorts": n_shorts,
        "win_rate": float(win_rate),
        "avg_trade": float(avg_trade),
        "trade_std": float(trade_std) if np.isfinite(trade_std) else float("nan"),
        "exposure": float(exposure),
        "dd_duration": float(max_dd_dur),
        "regime_filtered_n": float(regime_filtered_n),
        "regime_checks_n": float(regime_checks_n),
        "regime_filtered_pct": float(regime_filtered_pct),
        "max_fav_pct": float(max_fav_pct),
        "n_days": float(n),
    }
    if return_trades:
        out["trades"] = trade_logs
    return out

def infer_ga_ranges(score_z: np.ndarray) -> Tuple[Tuple[float, float], Tuple[float, float]]:
    x = np.abs(score_z[np.isfinite(score_z)])
    if len(x) < 200:
        return (0.15, 1.80), (0.04, 0.80)

    q30 = float(np.quantile(x, 0.30))
    q80 = float(np.quantile(x, 0.80))
    q92 = float(np.quantile(x, 0.92))

    enter_lo = max(0.08, q30 * 0.85)
    enter_hi = max(enter_lo * 2.5, q92 * 1.2, 0.60)

    exit_lo = 0.40
    exit_hi = max(exit_lo, min(0.55, enter_lo * 0.95))
    return (enter_lo, enter_hi), (exit_lo, exit_hi)



def ga_optimize_strategy_only(o, h, l, c, score_ev, ma, sl, atr, train_idx: np.ndarray, train_dates: np.ndarray = None, ticker: str = ""):
    o_tr = o[train_idx]; h_tr = h[train_idx]; l_tr = l[train_idx]; c_tr = c[train_idx]
    z_tr = score_ev[train_idx]
    ma_tr = ma[train_idx]; sl_tr = sl[train_idx]; atr_tr = atr[train_idx]
    d_tr = train_dates[train_idx] if train_dates is not None else None

    if np.isfinite(z_tr).sum() < 120:
        return None

    n_tr = len(c_tr)
    n_splits_ga = min(GA_WF_SPLITS, max(2, n_tr // 160))

    ga_fold_indices = []
    if n_tr >= 240:
        tscv_ga = TimeSeriesSplit(n_splits=n_splits_ga)
        for tr_idx, te_idx in tscv_ga.split(c_tr):
            if len(te_idx) >= 80 and len(tr_idx) >= 120:
                ga_fold_indices.append(te_idx)

    fitness_cache = {}

    def fitness_internal(p_: Params) -> float:
        p_ = sanitize_params(p_)
        key = (int(round(p_.fast_period)), int(round(p_.slow_period)), round(p_.atr_mult, 3), round(p_.rr_mult, 3), round(p_.entry_discount, 3))
        if key in fitness_cache:
            return fitness_cache[key]

        def _score_fold(st: Dict[str, float]) -> float:
            n_trades = float(st.get("n_trades", 0.0))
            n_days = float(st.get("n_days", np.nan))
            dyn_min_trades = float(max(4.0, min(float(GA_MIN_TRADES_FOR_SIGNIFICANCE), 0.03 * max(n_days, 1.0)))) if np.isfinite(n_days) else float(GA_MIN_TRADES_FOR_SIGNIFICANCE)

            ret = float(st.get("total_return", 0.0))
            mdd = float(st.get("mdd", 0.0))
            avg_trade = float(st.get("avg_trade", 0.0))
            trade_std = float(st.get("trade_std", np.nan))

            if (not np.isfinite(trade_std)) or trade_std <= 1e-9:
                sqn = -0.5
            else:
                sqn = (math.sqrt(max(n_trades, 1.0)) * avg_trade) / (trade_std + 1e-9)

            trade_participation = float(np.clip(n_trades / max(dyn_min_trades, 1.0), 0.0, 1.5))
            low_trade_penalty = -8.0 * max(0.0, 1.0 - trade_participation)
            neg_ret_penalty = -8.0 * max(0.0, -ret)
            deep_mdd_penalty = -4.0 * max(0.0, abs(mdd) - 0.20)

            score = (1.10 * ret) + (0.85 * sqn)
            score += min(n_trades, 80.0) * 0.02
            score += low_trade_penalty + neg_ret_penalty + deep_mdd_penalty

            if n_trades > GA_OVERTRADING_TRADES_PER_FOLD:
                score *= 0.80

            return float(score)

        if n_tr < 240:
            st = backtest_stats_only_intraday(o_tr, h_tr, l_tr, c_tr, z_tr, ma_tr, sl_tr, atr_tr, p_)
            st = dict(st)
            st["total_return"] = float(st.get("total_return", 0.0) - (GA_INTERNAL_EXTRA_TRADE_COST_BPS / 10000.0) * float(st.get("n_trades", 0.0)))
            val = _score_fold(st)
            fitness_cache[key] = val
            return val

        scores_folds = []
        for te_idx in ga_fold_indices:
            st = backtest_stats_only_intraday(
                o_tr[te_idx], h_tr[te_idx], l_tr[te_idx], c_tr[te_idx], z_tr[te_idx],
                ma_tr[te_idx], sl_tr[te_idx], atr_tr[te_idx], p_
            )
            st = dict(st)
            st["total_return"] = float(st.get("total_return", 0.0) - (GA_INTERNAL_EXTRA_TRADE_COST_BPS / 10000.0) * float(st.get("n_trades", 0.0)))
            scores_folds.append(_score_fold(st))

        if len(scores_folds) == 0:
            fitness_cache[key] = -1e9
            return -1e9

        avg_score = float(np.mean(scores_folds))
        stability_penalty = float(np.std(scores_folds))
        val = float(avg_score - (GA_FOLD_STABILITY_PENALTY * stability_penalty))
        fitness_cache[key] = val
        return val


    toolbox = base.Toolbox()
    toolbox.register("attr_fast", random.randint, FAST_PERIOD_RANGE[0], FAST_PERIOD_RANGE[1])
    toolbox.register("attr_slow", random.randint, SLOW_PERIOD_RANGE[0], SLOW_PERIOD_RANGE[1])
    toolbox.register("attr_atr",   random.uniform, ATR_MULT_RANGE[0], ATR_MULT_RANGE[1])
    toolbox.register("attr_rr",    random.uniform, RR_MULT_RANGE[0],  RR_MULT_RANGE[1])
    toolbox.register("attr_entry_discount", random.uniform, ENTRY_DISCOUNT_RANGE[0], ENTRY_DISCOUNT_RANGE[1])

    toolbox.register("individual", tools.initCycle, creator.Individual_PT,
                     (toolbox.attr_fast, toolbox.attr_slow, toolbox.attr_atr, toolbox.attr_rr, toolbox.attr_entry_discount), n=1)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("select", tools.selTournament, tournsize=GA_TOURN)
    toolbox.register("mate", tools.cxSimulatedBinary, eta=20.0)

    def _clip_ind(ind):
        ind[0] = float(int(np.clip(round(ind[0]), FAST_PERIOD_RANGE[0], FAST_PERIOD_RANGE[1])))
        ind[1] = float(int(np.clip(round(ind[1]), SLOW_PERIOD_RANGE[0], SLOW_PERIOD_RANGE[1])))
        if ind[1] <= ind[0]:
            ind[1] = float(min(SLOW_PERIOD_RANGE[1], int(ind[0]) + 1))
            if ind[1] <= ind[0]:
                ind[0] = float(max(FAST_PERIOD_RANGE[0], int(ind[1]) - 1))
        ind[2] = float(np.clip(ind[2], ATR_MULT_RANGE[0], ATR_MULT_RANGE[1]))
        ind[3] = float(np.clip(ind[3], RR_MULT_RANGE[0], RR_MULT_RANGE[1]))
        ind[4] = float(np.clip(ind[4], ENTRY_DISCOUNT_RANGE[0], ENTRY_DISCOUNT_RANGE[1]))
        return ind

    def mutate_gaussian(ind, sigma=0.10, gene_pb=0.20):
        for j in range(len(ind)):
            if random.random() < gene_pb:
                ind[j] = float(ind[j]) + random.gauss(0.0, sigma)
        _clip_ind(ind)
        return (ind,)

    toolbox.register("mutate", mutate_gaussian)
    toolbox.register("evaluate", lambda ind: (fitness_internal(Params(*ind)),))

    pop = toolbox.population(n=GA_POP_SIZE)
    hof = tools.HallOfFame(GA_HOF_SIZE)

    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    hof.update(pop)

    best_fit = hof[0].fitness.values[0]
    no_improve = 0

    verbose_gen = bool(GA_VERBOSE_PER_GENERATION and train_dates is not None)

    for _gen in range(1, GA_NGEN + 1):
        frac = (_gen - 1) / max(1, (GA_NGEN - 1))
        sigma = GA_MUT_SIGMA_START + (GA_MUT_SIGMA_END - GA_MUT_SIGMA_START) * frac
        gene_pb = GA_GENE_MUT_PB_START + (GA_GENE_MUT_PB_END - GA_GENE_MUT_PB_START) * frac

        offspring = [toolbox.clone(ind) for ind in toolbox.select(pop, len(pop))]

        for i in range(1, len(offspring), 2):
            if random.random() < GA_CX_PB:
                toolbox.mate(offspring[i-1], offspring[i])
                _clip_ind(offspring[i-1]); _clip_ind(offspring[i])
                del offspring[i-1].fitness.values, offspring[i].fitness.values

        for i in range(len(offspring)):
            if random.random() < GA_MUT_PB:
                toolbox.mutate(offspring[i], sigma=sigma, gene_pb=gene_pb)
                del offspring[i].fitness.values

        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)
        pop[:] = offspring
        hof.update(pop)

        cur = hof[0].fitness.values[0]
        if cur > best_fit + 1e-9:
            best_fit = cur
            no_improve = 0
        else:
            no_improve += 1

        if verbose_gen:
            p_gen = sanitize_params(Params(*hof[0]))
            st_gen = backtest_stats_only_intraday(o_tr, h_tr, l_tr, c_tr, z_tr, ma_tr, sl_tr, atr_tr, p_gen, return_trades=True, dates=d_tr)
            n_long = int(sum(1 for t in st_gen.get("trades", []) if t.get("side") == "LONG"))
            n_short = int(sum(1 for t in st_gen.get("trades", []) if t.get("side") == "SHORT"))
            print(
                f"[GA GEN {ticker}] gen={_gen:02d} fit={cur:.4f} best=(fast={int(p_gen.fast_period)}, slow={int(p_gen.slow_period)}, atr={p_gen.atr_mult:.3f}, rr={p_gen.rr_mult:.3f}, disc={p_gen.entry_discount:.3f}) "
                f"ret={100.0*float(st_gen.get('total_return', np.nan)):.2f}% mdd={100.0*float(st_gen.get('mdd', np.nan)):.2f}% sh={float(st_gen.get('sharpe', np.nan)):.2f} "
                f"trades={int(st_gen.get('n_trades', 0))} (compra={n_long}, venda={n_short})"
            )
            trades = st_gen.get("trades", [])
            for j, tr in enumerate(trades[:GA_VERBOSE_MAX_TRADES_PER_GEN], start=1):
                side_pt = "COMPRA" if tr.get('side') == 'LONG' else ("VENDA" if tr.get('side') == 'SHORT' else str(tr.get('side','?')))
                print(
                    f"    trade#{j:02d} {side_pt}/{tr.get('side','?')} {tr.get('entry_date')}->{tr.get('exit_date')} "
                    f"entry={float(tr.get('entry_price', np.nan)):.4f} exit={float(tr.get('exit_price', np.nan)):.4f} "
                    f"trigger={tr.get('entry_trigger','')} exit_reason={tr.get('exit_reason','')} "
                    f"gross={100.0*float(tr.get('gross_ret', np.nan)):.2f}% net={100.0*float(tr.get('net_ret', np.nan)):.2f}% "
                    f"bars={int(tr.get('bars_held', 0))} mfe={100.0*float(tr.get('max_fav_pct', np.nan)):.2f}%"
                )
            if len(trades) > GA_VERBOSE_MAX_TRADES_PER_GEN:
                print(f"    ... {len(trades)-GA_VERBOSE_MAX_TRADES_PER_GEN} trades omitted")

        if no_improve >= EARLY_STOP:
            break

    best_p = sanitize_params(Params(*hof[0]))

    start = max(0, len(c_tr) - ONE_YEAR_DAYS)
    stats_train = backtest_stats_only_intraday(
        o_tr[start:], h_tr[start:], l_tr[start:], c_tr[start:], z_tr[start:],
        ma_tr[start:], sl_tr[start:], atr_tr[start:], best_p
    )
    fit1y = fitness_return_1y(stats_train)
    bh1y  = buyhold_capped(c_tr[start:])

    return best_p, stats_train, fit1y, bh1y


def aggregate_window_stats(stats_list: List[Dict[str, float]]) -> Dict[str, float]:
    if not stats_list:
        return {
            "total_return": float("nan"), "mdd": float("nan"), "sharpe": float("nan"),
            "sortino": float("nan"), "n_trades": float("nan"), "win_rate": float("nan"),
            "avg_trade": float("nan"), "trade_std": float("nan"), "exposure": float("nan"),
            "dd_duration": float("nan"), "regime_filtered_pct": float("nan"), "max_fav_pct": float("nan"),
        }

    keys = set()
    for st in stats_list:
        if isinstance(st, dict):
            keys.update(st.keys())

    out = {}
    for k in keys:
        vals = [float(st.get(k, np.nan)) for st in stats_list if isinstance(st, dict)]
        vals = np.asarray(vals, dtype=np.float64)
        vals = vals[np.isfinite(vals)]
        out[k] = float(np.mean(vals)) if len(vals) else float("nan")

    return out


def build_ga_walkforward_windows(has_oos_idx: np.ndarray) -> List[Tuple[np.ndarray, np.ndarray]]:
    windows = []
    n = len(has_oos_idx)
    tr_len = GA_WF_TRAIN_YEARS * ONE_YEAR_DAYS
    te_len = GA_WF_TEST_DAYS
    step = GA_WF_STEP_DAYS
    if n < (tr_len + te_len):
        return windows
    end = tr_len
    while (end + te_len) <= n:
        tr_idx = has_oos_idx[end - tr_len:end]
        te_idx = has_oos_idx[end:end + te_len]
        windows.append((tr_idx, te_idx))
        end += step
    return windows


# ==============================================================================
# 7) LOAD
# ==============================================================================
def calibrate_ev_with_realized_feedback(
    score_ev: np.ndarray,
    y_atr_norm: np.ndarray,
    valid_mask: np.ndarray,
) -> Tuple[np.ndarray, Dict[str, float]]:
    out = score_ev.copy()
    meta = {"ic": float("nan"), "n_calib": 0.0, "bins_used": 0.0, "blend_used": 0.0}

    if (not USE_RETURN_FEEDBACK_CALIBRATION) or (RETURN_FEEDBACK_BLEND >= 1.0):
        return out, meta

    calib_mask = np.isfinite(score_ev) & np.isfinite(y_atr_norm) & valid_mask
    n_calib = int(np.sum(calib_mask))
    meta["n_calib"] = float(n_calib)
    if n_calib < RETURN_FEEDBACK_MIN_ROWS:
        return out, meta

    ev_cal = score_ev[calib_mask]
    y_cal = y_atr_norm[calib_mask]

    try:
        ic = float(np.corrcoef(ev_cal, y_cal)[0, 1])
    except Exception:
        ic = float("nan")
    if np.isfinite(ic):
        meta["ic"] = ic
    if (not np.isfinite(ic)) or (ic < RETURN_FEEDBACK_MIN_IC):
        return out, meta

    ic_strength = float(np.clip((ic - RETURN_FEEDBACK_MIN_IC) / max(1e-12, RETURN_FEEDBACK_TARGET_IC - RETURN_FEEDBACK_MIN_IC), 0.0, 1.0))
    blend_used = float(np.clip(RETURN_FEEDBACK_BLEND_MAX * ic_strength, 0.0, RETURN_FEEDBACK_BLEND_MAX))
    meta["blend_used"] = blend_used
    if blend_used <= 0.0:
        return out, meta

    try:
        bins = pd.qcut(ev_cal, q=RETURN_FEEDBACK_BINS, labels=False, duplicates="drop")
    except Exception:
        return out, meta

    bins = np.asarray(bins, dtype=float)
    if int(np.isfinite(bins).sum()) < RETURN_FEEDBACK_MIN_ROWS:
        return out, meta
    bins = bins.astype(int)
    n_bins = int(np.max(bins)) + 1
    if n_bins < 3:
        return out, meta
    meta["bins_used"] = float(n_bins)

    means = np.full(n_bins, np.nan, dtype=np.float64)
    global_mean = float(np.nanmean(y_cal)) if np.isfinite(np.nanmean(y_cal)) else 0.0
    for b in range(n_bins):
        m = (bins == b)
        if np.any(m):
            n_b = int(np.sum(m))
            raw_b = float(np.nanmean(y_cal[m]))
            shrink = float(n_b / (n_b + RETURN_FEEDBACK_BIN_SHRINK))
            means[b] = shrink * raw_b + (1.0 - shrink) * global_mean

    # enforce monotonic mapping to reduce overfitting noise (higher EV bin => no lower mapped return)
    means = np.maximum.accumulate(means)

    ev_all_mask = np.isfinite(score_ev)
    ev_all = score_ev[ev_all_mask]
    if len(ev_all) < 10:
        return out, meta

    ranks = pd.Series(ev_all).rank(method="average", pct=True).to_numpy()
    ids = np.minimum((ranks * n_bins).astype(int), n_bins - 1)
    mapped = means[ids]

    std_m = float(np.nanstd(mapped))
    std_e = float(np.nanstd(ev_all))
    if (not np.isfinite(std_m)) or (std_m <= 1e-12) or (not np.isfinite(std_e)) or (std_e <= 1e-12):
        return out, meta

    mapped = mapped * (std_e / std_m)
    out[ev_all_mask] = ((1.0 - blend_used) * ev_all) + (blend_used * mapped)
    return out, meta




def build_direct_feature_signal(frame: pd.DataFrame, feat_cols: List[str]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Use all selected FINAL columns directly (no ML) to generate directional pressure."""
    feat = frame[feat_cols].apply(pd.to_numeric, errors="coerce").copy()
    roll_mean = feat.rolling(SCORE_LOOKBACK, min_periods=60).mean()
    roll_std = feat.rolling(SCORE_LOOKBACK, min_periods=60).std(ddof=0).replace(0.0, np.nan)
    z = (feat - roll_mean) / roll_std
    z = z.replace([np.inf, -np.inf], np.nan).clip(-4.0, 4.0)

    bearish_tokens = ("risk", "down", "dd", "sell", "err_buy", "pe", "price_to_book")
    signs = []
    for c in feat_cols:
        lc = str(c).lower()
        sign = -1.0 if any(tok in lc for tok in bearish_tokens) else 1.0
        signs.append(sign)
    sign_arr = np.asarray(signs, dtype=np.float64)

    z_np = z.to_numpy(np.float64)
    directed = z_np * sign_arr
    long_votes = np.nanmean((directed > 0.35).astype(np.float64), axis=1)
    short_votes = np.nanmean((directed < -0.35).astype(np.float64), axis=1)
    pressure = np.nan_to_num(long_votes - short_votes, nan=0.0)
    return pressure.astype(np.float64), np.nan_to_num(long_votes, nan=0.0), np.nan_to_num(short_votes, nan=0.0)
def load_full_history_all_cols(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={TICKER_COL:"string"}, low_memory=False)
    df[TICKER_COL] = df[TICKER_COL].astype("string").str.strip()
    df[DATE_COL] = _parse_dates_smart(df[DATE_COL])

    for col in [OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[DATE_COL, TICKER_COL, OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL])
    df = df[(df[CLOSE_COL] > MIN_PRICE) & (df[OPEN_COL] > MIN_PRICE) & (df[HIGH_COL] > MIN_PRICE) & (df[LOW_COL] > MIN_PRICE)]
    if ONLY_SA:
        df = df[df[TICKER_COL].str.endswith(".SA", na=False)]

    df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)
    df = add_sma200(df)
    df = add_atr_ohlc_fast(df)
    df = add_technical_features(df)
    return df


# ==============================================================================
# 8) RUN
# ==============================================================================
def _pct(x):
    return f"{x*100:7.1f}%" if np.isfinite(x) else "    nan "
def _flt(x, w=6, p=3):
    return f"{x:{w}.{p}f}" if np.isfinite(x) else f"{'nan':>{w}}"

def run():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    df = load_full_history_all_cols(HISTORY_CSV_PATH)

    exclude = {DATE_COL, TICKER_COL, OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL, "sma200", "sma200_slope", "atr"}
    num_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    base_feat_cols = num_cols[:MAX_FEATURES]

    tickers = df[TICKER_COL].dropna().unique().tolist()
    total_tickers = len(tickers)

    if EVAL_ONLY_TICKER:
        t_req = str(EVAL_ONLY_TICKER).strip().upper()
        tickers = [t for t in tickers if str(t).upper() in {t_req, f"{t_req}.SA"}]
    elif TEST_ONLY_PREFIX_C_TICKERS:
        tickers = [t for t in tickers if str(t).startswith(TEST_TICKER_PREFIX)]

    print(f"[FEATS] numeric candidates: {len(base_feat_cols)} (using up to {MAX_FEATURES})")
    if EVAL_ONLY_TICKER:
        print(f"[TICKERS] processing only '{EVAL_ONLY_TICKER}': {len(tickers)} of {total_tickers}")
    elif TEST_ONLY_PREFIX_C_TICKERS:
        print(f"[TICKERS] processing prefix '{TEST_TICKER_PREFIX}': {len(tickers)} of {total_tickers}")
    else:
        print(f"[TICKERS] total: {len(tickers)}")

    if PRINT_ML_METRICS:
        header = (
            f"{'#':>4} | {'TICKER':<10} | "
            f"{'AUC':>6} {'AUCsd':>6} {'ACC':>5} {'AP':>5} | "
            f"{'GAret':>7} {'GAmdd':>7} {'GAsh':>6} {'TR':>4} {'EX':>4} {'BH':>7} || "
            f"{'TEret':>7} {'TEmdd':>7} {'TEsh':>6} {'TEtr':>4} {'Win%':>6} {'AvgTrd':>7} {'MaxFav%':>8} {'Exp':>5} {'DD_Dur':>7} {'RegFl%':>7} | {'TRAIN_RANGE':<24} {'TEST_RANGE':<24}"
        )
    else:
        header = (
            f"{'#':>4} | {'TICKER':<10} | "
            f"{'GAret':>7} {'GAmdd':>7} {'GAsh':>6} {'TR':>4} {'EX':>4} {'BH':>7} || "
            f"{'TEret':>7} {'TEmdd':>7} {'TEsh':>6} {'TEtr':>4} {'Win%':>6} {'AvgTrd':>7} {'MaxFav%':>8} {'Exp':>5} {'DD_Dur':>7} {'RegFl%':>7} | {'TRAIN_RANGE':<24} {'TEST_RANGE':<24}"
        )
    print("\n" + header + "\n" + "-"*len(header))

    reasons: Dict[str, int] = {}
    results_summary: List[Dict] = []
    results_apply: List[Dict] = []
    prog = 0

    for tkr in tickers:
        g = df[df[TICKER_COL] == tkr].copy().sort_values(DATE_COL)

        if len(g) < MIN_ROWS_TICKER:
            reasons["short_len"] = reasons.get("short_len", 0) + 1
            continue

        # Intelligent per-ticker feature selection using Spearman correlation
        # First, compute significant-event target to use in feature selection
        c_temp = g[CLOSE_COL].to_numpy(np.float64)
        atr_temp = g["atr"].to_numpy(np.float64)
        ret_fwd_temp = (np.roll(c_temp, -FWD_H) - c_temp) / np.maximum(c_temp, 1e-12)
        ret_fwd_temp[-FWD_H:] = np.nan
        atr_pct_temp = atr_temp / np.maximum(c_temp, 1e-12)
        thr_temp = np.maximum(TARGET_RET_THRESHOLD, TARGET_ATR_MULT * atr_pct_temp)
        y_event_temp = (ret_fwd_temp > thr_temp).astype(float)
        y_event_temp[~np.isfinite(ret_fwd_temp)] = np.nan
        y_event_temp[~np.isfinite(thr_temp)] = np.nan
        
        # Filter candidate features and compute correlations
        feat_correlations: List[tuple] = []
        for ccol in base_feat_cols:
            s = pd.to_numeric(g[ccol], errors="coerce")
            if float(s.notna().mean()) < MIN_FEAT_NONNA_FRAC:
                continue
            if float(s.std(skipna=True)) <= MIN_FEAT_STD:
                continue
            
            # Compute Spearman correlation with target
            try:
                valid_idx_temp = np.isfinite(s.to_numpy()) & np.isfinite(y_event_temp)
                if valid_idx_temp.sum() > MIN_VALID_SAMPLES_FOR_CORRELATION:  # Need enough valid samples
                    corr, _ = spearmanr(s.to_numpy()[valid_idx_temp], y_event_temp[valid_idx_temp])
                    if np.isfinite(corr):
                        feat_correlations.append((ccol, abs(corr)))
            except ValueError:
                pass
        
        # Select top MAX_FEATURES by absolute correlation
        feat_correlations.sort(key=lambda x: x[1], reverse=True)
        feat_cols: List[str] = [col for col, _ in feat_correlations[:MAX_FEATURES]]

        if len(feat_cols) < 5:
            reasons["few_feats"] = reasons.get("few_feats", 0) + 1
            continue

        # arrays
        dates = pd.to_datetime(g[DATE_COL], errors="coerce").to_numpy()
        o = g[OPEN_COL].to_numpy(np.float64)
        h = g[HIGH_COL].to_numpy(np.float64)
        l = g[LOW_COL].to_numpy(np.float64)
        c = g[CLOSE_COL].to_numpy(np.float64)
        ma  = g["sma200"].to_numpy(np.float64)
        sl  = g["sma200_slope"].to_numpy(np.float64)
        atr = g["atr"].to_numpy(np.float64)

        # Direct GA signal from FINAL columns (no ML score model)
        score_full, long_votes, short_votes = build_direct_feature_signal(g, feat_cols)

        valid_mask = np.isfinite(o) & np.isfinite(h) & np.isfinite(l) & np.isfinite(c) & np.isfinite(atr) & np.isfinite(score_full)
        if valid_mask.sum() < ML_MIN_TRAIN:
            reasons["few_valid"] = reasons.get("few_valid", 0) + 1
            continue

        score_ev_raw = np.clip(score_full, -1.0, 1.0)
        score_ev = pd.Series(score_ev_raw).ewm(span=EV_EMA_SPAN, adjust=False, min_periods=1).mean().to_numpy(np.float64)
        score_ev = np.clip(score_ev, -1.0, 1.0)
        score_ev_apply = score_ev.copy()
        feedback_meta = {"ic": float("nan"), "n_calib": 0.0, "bins_used": 0.0}
        tail_rows_predicted = 0

        wf_m = {
            "wf_auc_mean": float("nan"), "wf_auc_std": float("nan"), "wf_acc_mean": float("nan"),
            "wf_ap_mean": float("nan"), "wf_logloss": float("nan"), "wf_brier": float("nan"),
            "wf_fold_ranges": "",
        }
        wf_quality = 0.5

        has_ev_idx = np.where(np.isfinite(score_ev))[0]
        if len(has_ev_idx) < 252:
            reasons["short_oos"] = reasons.get("short_oos", 0) + 1
            continue

        has_oos_idx = np.where(valid_mask)[0]
        if len(has_oos_idx) < 252:
            has_oos_idx = has_ev_idx

        ga_windows = build_ga_walkforward_windows(has_oos_idx)
        # rolling walk-forward windows (train->next test) to approximate live re-optimization
        ga_train_window_stats = []
        ga_test_window_stats = []
        ga_window_fits = []
        ga_window_bh = []
        best_p = None

        if ga_windows:
            for w_tr, w_te in ga_windows:
                out_ga = ga_optimize_strategy_only(o, h, l, c, score_ev, ma, sl, atr, w_tr, dates, tkr)
                if out_ga is None:
                    continue
                p_w, st_w, fit_w, bh_w = out_ga
                p_w = adjust_params_by_wf_quality(p_w, wf_quality)
                te_w = backtest_stats_only_intraday(
                    o[w_te], h[w_te], l[w_te], c[w_te],
                    score_ev[w_te], ma[w_te], sl[w_te], atr[w_te],
                    p_w
                )
                ga_train_window_stats.append(st_w)
                ga_test_window_stats.append(te_w)
                ga_window_fits.append(float(fit_w))
                ga_window_bh.append(float(bh_w))
                best_p = p_w  # keep most recent window params for apply/live

        if (best_p is None) or (len(ga_test_window_stats) == 0):
            # fallback to static split when not enough windows
            split_point = int(len(has_oos_idx) * 0.80)
            ga_train_idx = has_oos_idx[:split_point]
            test_final_idx = has_oos_idx[split_point:]
            out_ga = ga_optimize_strategy_only(o, h, l, c, score_ev, ma, sl, atr, ga_train_idx, dates, tkr)
            if out_ga is None:
                reasons["ga_fail"] = reasons.get("ga_fail", 0) + 1
                continue
            best_p, stats_ga_train, fit_ga, bh_ga = out_ga
            best_p = adjust_params_by_wf_quality(best_p, wf_quality)
            te_stats = backtest_stats_only_intraday(
                o[test_final_idx], h[test_final_idx], l[test_final_idx], c[test_final_idx],
                score_ev[test_final_idx], ma[test_final_idx], sl[test_final_idx], atr[test_final_idx],
                best_p
            )
            train_rng = (pd.to_datetime(dates[ga_train_idx[0]]).date(), pd.to_datetime(dates[ga_train_idx[-1]]).date())
            test_rng  = (pd.to_datetime(dates[test_final_idx[0]]).date(), pd.to_datetime(dates[test_final_idx[-1]]).date())
        else:
            te_stats = aggregate_window_stats(ga_test_window_stats)
            stats_ga_train = aggregate_window_stats(ga_train_window_stats)
            fit_ga = float(np.mean(ga_window_fits)) if ga_window_fits else float("nan")
            bh_ga = float(np.mean(ga_window_bh)) if ga_window_bh else float("nan")
            ga_train_idx, test_final_idx = ga_windows[-1]
            train_rng = (pd.to_datetime(dates[ga_windows[-1][0][0]]).date(), pd.to_datetime(dates[ga_windows[-1][0][-1]]).date())
            test_rng  = (pd.to_datetime(dates[ga_windows[-1][1][0]]).date(), pd.to_datetime(dates[ga_windows[-1][1][-1]]).date())

        best_p_adj = best_p

        prog += 1
        if (prog % PRINT_EVERY) == 0:
            if PRINT_ML_METRICS:
                print(
                    f"{prog:4d} | {tkr:<10} | "
                    f"{_flt(wf_m['wf_auc_mean'],6,3)} {_flt(wf_m['wf_auc_std'],6,3)} {_flt(wf_m['wf_acc_mean'],5,3)} {_flt(wf_m['wf_ap_mean'],5,3)} | "
                    f"{_pct(stats_ga_train['total_return'])} {_pct(stats_ga_train['mdd'])} {_flt(stats_ga_train['sharpe'],6,2)} "
                    f"{int(stats_ga_train['n_trades']):4d} {int(stats_ga_train['exposure']*100):3d}% {_pct(bh_ga)} || "
                    f"{_pct(te_stats.get('total_return', np.nan))} {_pct(te_stats.get('mdd', np.nan))} {_flt(te_stats.get('sharpe', np.nan),6,2)} "
                    f"{int(te_stats.get('n_trades', 0) if np.isfinite(te_stats.get('n_trades', np.nan)) else 0):4d} "
                    f"{100.0*float(te_stats.get('win_rate', np.nan)):5.1f}% "
                    f"{100.0*float(te_stats.get('avg_trade', np.nan)):6.2f}% "
                    f"{100.0*float(te_stats.get('max_fav_pct', np.nan)):7.2f}% "
                    f"{100.0*float(te_stats.get('exposure', np.nan)):4.0f}% "
                    f"{int(te_stats.get('dd_duration', 0) if np.isfinite(te_stats.get('dd_duration', np.nan)) else 0):7d} "
                    f"{100.0*float(te_stats.get('regime_filtered_pct', 0.0)):6.1f}% | "
                    f"{str(train_rng[0])+' -> '+str(train_rng[1]):<24} {str(test_rng[0])+' -> '+str(test_rng[1]):<24}"
                )
                if USE_RETURN_FEEDBACK_CALIBRATION and np.isfinite(feedback_meta.get("ic", np.nan)):
                    print(f"      [ML-feedback] IC={feedback_meta['ic']:.3f} blend={feedback_meta.get('blend_used', 0.0):.2f} bins={int(feedback_meta.get('bins_used', 0))} n={int(feedback_meta.get('n_calib', 0))}")
            else:
                print(
                    f"{prog:4d} | {tkr:<10} | "
                    f"{_pct(stats_ga_train['total_return'])} {_pct(stats_ga_train['mdd'])} {_flt(stats_ga_train['sharpe'],6,2)} "
                    f"{int(stats_ga_train['n_trades']):4d} {int(stats_ga_train['exposure']*100):3d}% {_pct(bh_ga)} || "
                    f"{_pct(te_stats.get('total_return', np.nan))} {_pct(te_stats.get('mdd', np.nan))} {_flt(te_stats.get('sharpe', np.nan),6,2)} "
                    f"{int(te_stats.get('n_trades', 0) if np.isfinite(te_stats.get('n_trades', np.nan)) else 0):4d} "
                    f"{100.0*float(te_stats.get('win_rate', np.nan)):5.1f}% "
                    f"{100.0*float(te_stats.get('avg_trade', np.nan)):6.2f}% "
                    f"{100.0*float(te_stats.get('max_fav_pct', np.nan)):7.2f}% "
                    f"{100.0*float(te_stats.get('exposure', np.nan)):4.0f}% "
                    f"{int(te_stats.get('dd_duration', 0) if np.isfinite(te_stats.get('dd_duration', np.nan)) else 0):7d} "
                    f"{100.0*float(te_stats.get('regime_filtered_pct', 0.0)):6.1f}% | "
                    f"{str(train_rng[0])+' -> '+str(train_rng[1]):<24} {str(test_rng[0])+' -> '+str(test_rng[1]):<24}"
                )

        # APPLY: last APPLY_DAYS *decision days*; suggested trade is for next day (i+1)
        start_apply = max(0, len(g) - APPLY_DAYS)
        idx_apply = np.arange(start_apply, len(g))
        apply_ev_non_null_frac = float(np.isfinite(score_ev[idx_apply]).mean()) if len(idx_apply) else 0.0

        ticker_apply_rows = []
        for i in idx_apply:
            # EOD decision (day i)
            date_i = pd.to_datetime(dates[i])
            close_i = float(c[i])
            ev_i = float(score_ev_apply[i]) if np.isfinite(score_ev_apply[i]) else np.nan
            atr_i = float(atr[i]) if np.isfinite(atr[i]) else np.nan
            ma_i = float(ma[i]) if np.isfinite(ma[i]) else np.nan
            sl_i = float(sl[i]) if np.isfinite(sl[i]) else np.nan

            signal_eod = make_signal_eod(score_ev_apply, i, best_p_adj, close_i, ma_i, sl_i)
            final_signal = signal_eod
            quality = 0.60 * wf_quality + 0.40 * compute_quality_factor(float(te_stats.get("sharpe", np.nan)), float(te_stats.get("total_return", np.nan)), float(stats_ga_train.get("n_trades", np.nan)))
            recent_scores = score_ev_apply[max(0, i-SCORE_LOOKBACK): i+1]
            score100 = score_0_100_from_ev(ev_i, recent_scores, quality)

            # next day projections + fills (always compute projected limits when i+1 exists)
            best_buy_value = np.nan
            best_sell_value = np.nan
            limit_price = np.nan
            next_day_filled = False
            next_day_filled_buy = False
            next_day_filled_sell = False
            projected_buy_limit = np.nan
            projected_sell_limit = np.nan
            entry_ref_price = np.nan

            if (i + 1) < len(g):
                o1, h1, l1 = float(o[i+1]), float(h[i+1]), float(l[i+1])
                atr1 = float(atr[i+1]) if np.isfinite(atr[i+1]) else np.nan

                projected_buy_limit = compute_model_entry_price(o1, atr1, ev_i, SCORE_CROSS_MIN_ABS, +1, best_p_adj.entry_discount)
                projected_sell_limit = compute_model_entry_price(o1, atr1, ev_i, SCORE_CROSS_MIN_ABS, -1, best_p_adj.entry_discount)

                fb, fill_b, lim_b = nextday_limit_fill(o1, h1, l1, atr1, ev_i, SCORE_CROSS_MIN_ABS, +1, best_p_adj.entry_discount)
                fs, fill_s, lim_s = nextday_limit_fill(o1, h1, l1, atr1, ev_i, SCORE_CROSS_MIN_ABS, -1, best_p_adj.entry_discount)
                next_day_filled_buy = bool(fb)
                next_day_filled_sell = bool(fs)

                # Keep "best_*" always informative: projected limit if no fill, filled price if filled
                best_buy_value = float(fill_b) if fb else (float(lim_b) if np.isfinite(lim_b) else (float(projected_buy_limit) if np.isfinite(projected_buy_limit) else np.nan))
                best_sell_value = float(fill_s) if fs else (float(lim_s) if np.isfinite(lim_s) else (float(projected_sell_limit) if np.isfinite(projected_sell_limit) else np.nan))

                if final_signal == "buy":
                    next_day_filled = bool(fb)
                    limit_price = float(lim_b) if np.isfinite(lim_b) else np.nan
                    entry_ref_price = float(fill_b) if fb else (float(lim_b) if np.isfinite(lim_b) else float(o1))
                elif final_signal == "sell":
                    next_day_filled = bool(fs)
                    limit_price = float(lim_s) if np.isfinite(lim_s) else np.nan
                    entry_ref_price = float(fill_s) if fs else (float(lim_s) if np.isfinite(lim_s) else float(o1))
                else:
                    # For hold rows, still expose a meaningful reference for risk columns
                    entry_ref_price = float(o1)
            else:
                # last row has no next day: still provide fully-populated projected values
                ref_px = close_i
                atr_ref = atr_i if (np.isfinite(atr_i) and atr_i > ATR_EPS) else max(abs(close_i) * 0.01, ATR_EPS)
                projected_buy_limit = compute_model_entry_price(ref_px, atr_ref, ev_i, SCORE_CROSS_MIN_ABS, +1, best_p_adj.entry_discount)
                projected_sell_limit = compute_model_entry_price(ref_px, atr_ref, ev_i, SCORE_CROSS_MIN_ABS, -1, best_p_adj.entry_discount)
                best_buy_value = float(projected_buy_limit) if np.isfinite(projected_buy_limit) else ref_px
                best_sell_value = float(projected_sell_limit) if np.isfinite(projected_sell_limit) else ref_px
                entry_ref_price = ref_px
                limit_price = ref_px

            # compute levels from reference price and ATR of next day (if available)
            atr_for_levels = float(atr[i+1]) if (i + 1) < len(g) and np.isfinite(atr[i+1]) else atr_i
            levels = compute_levels_from_atr(entry_ref_price, atr_for_levels, best_p_adj)

            ticker_apply_rows.append({
                "Date": date_i,
                "ticker": tkr,
                "close": close_i,
                "atr": atr_i,

                "signal_eod": signal_eod,
                "signal": final_signal,

                "score_0_100": float(score100),
                "score_ev": ev_i,

                "wf_auc_mean": wf_m["wf_auc_mean"],
                "wf_auc_std": wf_m["wf_auc_std"],
                "wf_acc_mean": wf_m["wf_acc_mean"],
                "wf_ap_mean": wf_m["wf_ap_mean"],
                "wf_logloss": wf_m["wf_logloss"],
                "wf_brier": wf_m["wf_brier"],
            "wf_fold_ranges": wf_m["wf_fold_ranges"],

                "ga_fast_period": float(best_p_adj.fast_period),
                "ga_slow_period": float(best_p_adj.slow_period),
                "ga_enter_abs": float(best_p_adj.fast_period),
                "ga_exit_abs": float(best_p_adj.slow_period),
                "ga_atr_mult": float(best_p_adj.atr_mult),
                "ga_rr_mult": float(best_p_adj.rr_mult),
                "ga_entry_discount": float(best_p_adj.entry_discount),
                "wf_quality": float(wf_quality),
                "ml_ev_ic": float(feedback_meta.get("ic", np.nan)),
                "ml_feedback_n": float(feedback_meta.get("n_calib", 0.0)),
                "ml_feedback_blend": float(feedback_meta.get("blend_used", 0.0)),
                "tail_rows_predicted": int(tail_rows_predicted),
                "apply_ev_non_null_frac": float(apply_ev_non_null_frac),
                "ga_wf_windows": int(len(ga_windows)),
                "ga_wf_mode": "rolling" if len(ga_windows) > 0 else "static",

                "ga_return_1y": float(stats_ga_train["total_return"]),
                "ga_mdd_1y": float(stats_ga_train["mdd"]),
                "ga_sharpe_1y": float(stats_ga_train["sharpe"]),
                "ga_trades_1y": float(stats_ga_train["n_trades"]),
                "ga_exposure_1y": float(stats_ga_train["exposure"]),
                "buyhold_return_1y": float(bh_ga),

                "test_return": float(te_stats.get("total_return", np.nan)),
                "test_mdd": float(te_stats.get("mdd", np.nan)),
                "test_sharpe": float(te_stats.get("sharpe", np.nan)),
                "test_n_trades": float(te_stats.get("n_trades", np.nan)),
            "test_win_rate": float(te_stats.get("win_rate", np.nan)),
            "test_avg_trade": float(te_stats.get("avg_trade", np.nan)),
            "test_max_fav_pct": float(te_stats.get("max_fav_pct", np.nan)),
            "test_exposure": float(te_stats.get("exposure", np.nan)),
            "test_dd_duration": float(te_stats.get("dd_duration", np.nan)),
            "test_regime_filtered_pct": float(te_stats.get("regime_filtered_pct", np.nan)),

                "next_day_filled": bool(next_day_filled),
                "next_day_filled_buy": bool(next_day_filled_buy),
                "next_day_filled_sell": bool(next_day_filled_sell),
                "projected_buy_limit": float(projected_buy_limit) if np.isfinite(projected_buy_limit) else np.nan,
                "projected_sell_limit": float(projected_sell_limit) if np.isfinite(projected_sell_limit) else np.nan,
                "limit_price_next_day": float(limit_price) if np.isfinite(limit_price) else np.nan,
                "best_buy_value": float(best_buy_value) if np.isfinite(best_buy_value) else np.nan,
                "best_sell_value": float(best_sell_value) if np.isfinite(best_sell_value) else np.nan,
                "entry_ref_price": float(entry_ref_price) if np.isfinite(entry_ref_price) else np.nan,

                "stop_abs": levels[0], "take_abs": levels[1], "stop_pct": levels[2], "take_pct": levels[3],
                "buy_entry": levels[4], "buy_stop": levels[5], "buy_take": levels[6],
                "sell_entry": levels[7], "sell_stop": levels[8], "sell_take": levels[9],

                "train_start": train_rng[0], "train_end": train_rng[1],
                "test_start": test_rng[0], "test_end": test_rng[1],
            })


        results_apply.extend(ticker_apply_rows)

        # summary latest (at last available close)
        last_i = len(g) - 1
        latest_date  = pd.to_datetime(dates[last_i])
        latest_close = float(c[last_i])
        latest_atr   = float(atr[last_i]) if np.isfinite(atr[last_i]) else np.nan
        latest_z     = float(score_ev[last_i]) if np.isfinite(score_ev[last_i]) else np.nan
        latest_ma    = float(ma[last_i]) if np.isfinite(ma[last_i]) else np.nan
        latest_sl    = float(sl[last_i]) if np.isfinite(sl[last_i]) else np.nan

        signal_eod_latest = make_signal_eod(score_ev_apply, last_i, best_p_adj, latest_close, latest_ma, latest_sl)
        latest_quality = 0.60 * wf_quality + 0.40 * compute_quality_factor(float(te_stats.get("sharpe", np.nan)), float(te_stats.get("total_return", np.nan)), float(stats_ga_train.get("n_trades", np.nan)))
        latest_recent_scores = score_ev_apply[max(0, last_i-SCORE_LOOKBACK): last_i+1]
        latest_score100 = score_0_100_from_ev(latest_z, latest_recent_scores, latest_quality)
        final_signal_latest = signal_eod_latest

        results_summary.append({
            "ticker": tkr,
            "feat_count_used": int(len(feat_cols)),
            "wf_auc_mean": wf_m["wf_auc_mean"],
            "wf_auc_std": wf_m["wf_auc_std"],
            "wf_acc_mean": wf_m["wf_acc_mean"],
            "wf_ap_mean": wf_m["wf_ap_mean"],
            "wf_logloss": wf_m["wf_logloss"],
            "wf_brier": wf_m["wf_brier"],
            "wf_fold_ranges": wf_m["wf_fold_ranges"],

            "ga_fast_period": float(best_p_adj.fast_period),
            "ga_slow_period": float(best_p_adj.slow_period),
            "ga_enter_abs": float(best_p_adj.fast_period),
            "ga_exit_abs": float(best_p_adj.slow_period),
            "ga_atr_mult": float(best_p_adj.atr_mult),
            "ga_rr_mult": float(best_p_adj.rr_mult),
            "ga_entry_discount": float(best_p_adj.entry_discount),
            "wf_quality": float(wf_quality),
            "ga_return_1y": float(stats_ga_train["total_return"]),
            "ga_mdd_1y": float(stats_ga_train["mdd"]),
            "ga_sharpe_1y": float(stats_ga_train["sharpe"]),
            "ga_trades_1y": float(stats_ga_train["n_trades"]),
            "ga_exposure_1y": float(stats_ga_train["exposure"]),
            "ga_fitness_1y": float(fit_ga),
            "buyhold_return_1y": float(bh_ga),

            "test_return": float(te_stats.get("total_return", np.nan)),
            "test_mdd": float(te_stats.get("mdd", np.nan)),
            "test_sharpe": float(te_stats.get("sharpe", np.nan)),
            "test_n_trades": float(te_stats.get("n_trades", np.nan)),
            "test_win_rate": float(te_stats.get("win_rate", np.nan)),
            "test_avg_trade": float(te_stats.get("avg_trade", np.nan)),
            "test_max_fav_pct": float(te_stats.get("max_fav_pct", np.nan)),
            "test_exposure": float(te_stats.get("exposure", np.nan)),
            "test_dd_duration": float(te_stats.get("dd_duration", np.nan)),
            "test_regime_filtered_pct": float(te_stats.get("regime_filtered_pct", np.nan)),

            "latest_date": latest_date,
            "latest_close": latest_close,
            "latest_atr": latest_atr,
            "latest_score_ev": latest_z,
            "signal": final_signal_latest,
            "score_0_100": float(latest_score100),

            "train_start": train_rng[0], "train_end": train_rng[1],
            "test_start": test_rng[0], "test_end": test_rng[1],
        })

    if not results_summary:
        print("\nNenhum ticker gerou resultado.")
        if reasons:
            print("\n[DIAGNÓSTICO - motivos de descarte]")
            for k, v in sorted(reasons.items(), key=lambda x: -x[1]):
                print(f"  {k}: {v}")
        return

    summary_df = pd.DataFrame(results_summary).copy()
    summary_df["latest_date"] = pd.to_datetime(summary_df["latest_date"], errors="coerce")
    summary_df = summary_df.sort_values(["score_0_100", "ga_fitness_1y"], ascending=[False, False])

    apply_df = pd.DataFrame(results_apply).copy()
    apply_df["Date"] = pd.to_datetime(apply_df["Date"], errors="coerce")
    apply_df = apply_df.sort_values(["ticker", "Date"], ascending=[True, False])

    # Ensure output sheets are fully populated (avoid blank cells in Excel)
    bool_cols = ["next_day_filled", "next_day_filled_buy", "next_day_filled_sell"]
    for bc in bool_cols:
        if bc in apply_df.columns:
            apply_df[bc] = apply_df[bc].fillna(False).astype(bool)

    numeric_fill_zero = [
        "score_0_100", "score_ev", "wf_auc_mean", "wf_auc_std", "wf_acc_mean", "wf_ap_mean", "wf_logloss", "wf_brier", "wf_quality",
        "ga_fast_period", "ga_slow_period", "ga_enter_abs", "ga_exit_abs", "ga_atr_mult", "ga_rr_mult", "ga_entry_discount",
        "ga_return_1y", "ga_mdd_1y", "ga_sharpe_1y", "ga_trades_1y", "ga_exposure_1y", "buyhold_return_1y",
        "test_return", "test_mdd", "test_sharpe", "test_n_trades", "atr", "stop_abs", "take_abs", "stop_pct", "take_pct"
    ]
    for nc in numeric_fill_zero:
        if nc in apply_df.columns:
            apply_df[nc] = pd.to_numeric(apply_df[nc], errors="coerce").fillna(0.0)

    # price-like columns fallback to close to avoid blanks
    price_cols = [
        "best_buy_value", "best_sell_value", "projected_buy_limit", "projected_sell_limit", "limit_price_next_day",
        "entry_ref_price", "buy_entry", "buy_stop", "buy_take", "sell_entry", "sell_stop", "sell_take"
    ]
    for pc in price_cols:
        if pc in apply_df.columns:
            apply_df[pc] = pd.to_numeric(apply_df[pc], errors="coerce")
            apply_df[pc] = apply_df[pc].fillna(apply_df["close"])

    if "signal" in apply_df.columns:
        apply_df["signal"] = apply_df["signal"].fillna("hold")
    if "wf_fold_ranges" in apply_df.columns:
        apply_df["wf_fold_ranges"] = apply_df["wf_fold_ranges"].fillna("")
    for dc in ["train_start", "train_end", "test_start", "test_end"]:
        if dc in apply_df.columns:
            apply_df[dc] = apply_df[dc].fillna("")

    # user-friendly operational aliases
    if "test_return" in apply_df.columns:
        apply_df["TEret"] = apply_df["test_return"]
    if "test_mdd" in apply_df.columns:
        apply_df["TEmdd"] = apply_df["test_mdd"]
    if "stop_abs" in apply_df.columns:
        apply_df["stop_reais"] = apply_df["stop_abs"]
    if "take_abs" in apply_df.columns:
        apply_df["take_reais"] = apply_df["take_abs"]

    signals_cols = ["Date", "ticker", "close", "signal", "score_0_100", "TEret", "TEmdd", "best_buy_value", "best_sell_value", "stop_reais", "take_reais", "buy_stop", "buy_take", "sell_stop", "sell_take", "stop_pct", "take_pct", "atr"]
    debug_cols = ["Date", "ticker", "close", "signal", "score_0_100", "score_ev", "wf_auc_mean", "wf_auc_std", "wf_acc_mean", "wf_ap_mean", "wf_logloss", "wf_brier", "wf_fold_ranges", "wf_quality", "ml_ev_ic", "ml_feedback_n", "ml_feedback_blend", "tail_rows_predicted", "apply_ev_non_null_frac", "ga_wf_windows", "ga_wf_mode", "ga_fast_period", "ga_slow_period", "ga_enter_abs", "ga_exit_abs", "ga_atr_mult", "ga_rr_mult", "ga_entry_discount", "ga_return_1y", "ga_mdd_1y", "ga_sharpe_1y", "ga_trades_1y", "ga_exposure_1y", "buyhold_return_1y", "test_return", "test_mdd", "test_sharpe", "test_n_trades", "test_win_rate", "test_avg_trade", "test_max_fav_pct", "test_exposure", "test_dd_duration", "test_regime_filtered_pct", "best_buy_value", "best_sell_value", "projected_buy_limit", "projected_sell_limit", "next_day_filled", "next_day_filled_buy", "next_day_filled_sell", "limit_price_next_day", "entry_ref_price", "stop_abs", "take_abs", "train_start", "train_end", "test_start", "test_end"]
    signals_df = apply_df.reindex(columns=signals_cols)
    debug_df = apply_df.reindex(columns=debug_cols)

    out_xlsx = f"{OUTPUT_DIR}apply_PER_TICKER_WFGA_intraday__H{FWD_H}__APPLY{APPLY_DAYS}D__v2.xlsx"
    out_apply_csv = f"{OUTPUT_DIR}apply_last_{APPLY_DAYS}d__H{FWD_H}__v2.csv"

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        signals_df.to_excel(writer, sheet_name="signals", index=False)
        debug_df.to_excel(writer, sheet_name="debug_metrics", index=False)

    apply_df.to_csv(out_apply_csv, index=False, encoding="utf-8")

    print(f"\n[OK] Saved: {out_xlsx}")
    print(f"[OK] Saved CSV apply: {out_apply_csv}")

    print("\n[SINAIS - latest (final)]")
    print(summary_df["signal"].value_counts(dropna=False))

    ev_vals = pd.to_numeric(apply_df.get("score_ev"), errors="coerce").dropna()
    if len(ev_vals):
        print(f"\n[score_ev apply stats] mean={ev_vals.mean():.4f}, std={ev_vals.std():.4f}, min={ev_vals.min():.4f}, max={ev_vals.max():.4f}")
        print(f"[score_ev non-null frac] {(len(ev_vals)/max(len(apply_df),1)):.3f}")
    s100_vals = pd.to_numeric(apply_df.get("score_0_100"), errors="coerce").dropna()
    if len(s100_vals):
        print(f"[score_0_100 stats] mean={s100_vals.mean():.1f}, min={s100_vals.min():.1f}, max={s100_vals.max():.1f}")

    if reasons:
        print("\n[DIAGNÓSTICO - motivos de descarte]")
        for k, v in sorted(reasons.items(), key=lambda x: -x[1]):
            print(f"  {k}: {v}")


if __name__ == "__main__":
    run()






















